# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 제출 정보 (Chapter 05 제출 답안)

- 이름: 장태진
- GitHub ID: Taejin-AI
- 개인 저장소: `llm-data-analysis-study`
- 작성일: 2026.10.22
- 사용한 LLM: Claude (Anthropic)
- 실행 환경: Windows 11, 강사 저장소 공식 clone(`https://github.com/GilbertMoon/llm-data-analysis-course`, HEAD `6105f42`, 2026-10-19), 프로젝트 `.venv`(Python 3.13.1, `requirements.txt` 기준 pandas 3.0.5, numpy 2.5.3) 커널. 실제 경로와 버전은 아래 3번 셀 Output에 있습니다.
- 최종 제출 URL:

```text
https://github.com/Taejin-AI/llm-data-analysis-study/blob/master/chapter05/chapter05.ipynb
```

> 작성 방식: 공식 Notebook `notebooks/ch05_data_preprocessing.ipynb`를 복사해 1~25번 셀을 순서대로 그대로 실행했습니다. 실습 가이드 STEP의 Evidence와 답안 양식 항목은 `[추가 검증]` 코드 셀과 Markdown 셀로 사이에 추가했습니다. 공식 셀 중 수정한 것은 3번 경로 설정 셀 하나이고, 수정 내용은 셀 안에 `[수정]`으로 표시했습니다.

## 전처리 목적과 범위

- 목적: Chapter 06 EDA에서 "completed 주문 기준 월별·카테고리별 상품 주문 금액"을 계산할 수 있도록, 처리 기준과 전후 변화가 기록된 clean 데이터를 만듭니다.
- 입력 데이터: `practice/chapter05/data/raw`의 4개 CSV (Chapter 05 전용 Raw. 표기 차이, 변환 실패, 결측, 중복, 이상값, PK/FK 문제가 의도적으로 들어 있음)
  - 공식 Notebook의 기본 경로 `data/raw`는 결측·중복이 없는 공통 데이터입니다. 이 데이터로는 전처리 전후 차이를 확인할 수 없어서, `scripts/preprocess_data.py`와 같은 입력을 사용했습니다. `data/raw`는 읽지도 수정하지도 않았습니다.
- 출력: `data/processed/*_clean.csv` 4개, `reports/ch05_preprocessing_summary.md`
- 금액 용어: `line_total = quantity × unit_price`는 주문 상세 한 행의 상품 주문 금액입니다. 할인, 쿠폰, 배송비, 세금, 실제 결제액 정보가 없으므로 회계상 매출이나 순매출과 같다고 보지 않습니다.
- Chapter 06에서 쓸 테이블과 컬럼 (전처리 결과 기준으로 확인한 범위, 9번 섹션 `[추가 검증]` 셀):

| 필요한 것 | 테이블.컬럼 | 연결 키 |
| --- | --- | --- |
| 상품 주문 금액 | `order_items.quantity`, `order_items.unit_price` (→ `line_total`) | — |
| 월 구분, 분석 기간 | `orders.order_date` (→ `order_month`) | `order_items.order_id` → `orders.order_id` |
| completed 필터 | `orders.order_status` | 위와 같음 |
| 카테고리 | `products.category` | `order_items.product_id` → `products.product_id` |
| (고객 분석을 할 때만) 고객 속성 | `customers.age`, `customers.city` | `orders.customer_id` → `customers.customer_id` |

- 분석 기간: 유효 주문일 2025-07-01 ~ 2026-08-30 (변환 실패 3건과 작성일 2026-10-22 이후 주문 2건 제외). 이 기간의 completed 주문 176건, 주문 상세 421행, 상품 주문 금액 134,513,000이 월별·카테고리별 분석 대상입니다.
- 이번 질문(월별·카테고리별)에는 `customers.csv`가 필요하지 않습니다. 전처리는 4개 파일 모두 하지만, 고객 단위 분석을 할 때만 `customers`를 연결합니다. `customers.name`은 어떤 분석에도 필요하지 않아 출력하지 않았습니다.

## 실습 가이드 STEP과 이 Notebook의 위치

| 가이드 STEP | 이 Notebook 위치 | 답안 양식 |
| --- | --- | --- |
| 1 경로·환경 | 3번 (`[수정]` 셀) | 0 |
| 2 원본 4개 불러오기 | 4번 | — |
| 3 전처리 전 상태 | 5번 + `[추가 검증] STEP 3` | 1 |
| 4 복사본 | 6번 | — |
| 5 문자열 공백 | 10번 + `[추가 검증]` | 2 |
| 6 숫자·날짜 변환 + 실패 건수 | 12·13번 + `[추가 검증]` | 2 |
| 7 결측 처리 기준 | 7·8번 + `[추가 검증]` | 2 |
| 8 전체 중복·키 중복 | 9번 + `[추가 검증]` | 3 |
| 9 상태값 표준화 | 11번 + `[추가 검증]` | 3 |
| 10 이상값 후보 | 14번 + `[추가 검증]` 2개 | 3 |
| 11 파생 컬럼 | 12·15번 + `[추가 검증]` | 4 |
| 12 관계 재검증 | 16번 + `[추가 검증]` 2개 | 4 |
| 13 processed CSV 저장 | 18번 | 5 |
| 14 스크립트 재실행 | 22번 + `[추가 검증] STEP 14` | 6 |
| 15 저장 결과 재확인 | `[추가 검증] STEP 15` | 6 |
| 16 보고서 검토 | `[추가 검증] STEP 16` | 6 |
| 17 LLM 제안 검증 | 23번 + `[추가 검증]` 2개 | 7 앞 |

## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
# [수정] 공식 셀의 경로 설정을 두 군데 바꿨습니다.
# 1) 이 Notebook은 개인 저장소(llm-data-analysis-study/chapter05)에 있으므로,
#    형제 폴더인 강사 저장소(llm-data-analysis-course) 루트도 찾도록 했습니다.
# 2) RAW_DIR을 scripts/preprocess_data.py와 같은 Chapter 05 전용 Raw로 바꿨습니다.
from pathlib import Path
import hashlib
import os
import subprocess
import sys

import pandas as pd
import numpy as np


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        for root in (candidate, candidate / 'llm-data-analysis-course'):
            if (root / 'requirements.txt').exists() and (root / 'scripts').exists():
                return root
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

RAW_DIR = PROJECT_ROOT / 'practice' / 'chapter05' / 'data' / 'raw'   # [수정] 공식: data/raw
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)

git_head = subprocess.run(['git', 'log', '-1', '--format=%h %ci'], cwd=PROJECT_ROOT,
                          capture_output=True, text=True).stdout.strip()
git_remote = subprocess.run(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT_ROOT,
                            capture_output=True, text=True).stdout.strip()

print('Python 실행 파일:', sys.executable)
print('Python:', sys.version.split()[0], '/ pandas:', pd.__version__, '/ numpy:', np.__version__)
print('강사 저장소 clone:', git_remote, '| HEAD', git_head)
print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR, '| 존재:', RAW_DIR.exists())
print('전처리 데이터 폴더:', PROCESSED_DIR, '| 존재:', PROCESSED_DIR.exists())
print('보고서 폴더:', REPORT_DIR, '| 존재:', REPORT_DIR.exists())

Python 실행 파일: D:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
Python: 3.13.1 / pandas: 3.0.5 / numpy: 2.5.3
강사 저장소 clone: https://github.com/GilbertMoon/llm-data-analysis-course | HEAD 6105f42 2026-10-19 08:32:07 +0900
현재 실행 위치: D:\dev\llm-data-analysis-study\chapter05
프로젝트 루트: D:\dev\llm-data-analysis-course
원본 데이터 폴더: D:\dev\llm-data-analysis-course\practice\chapter05\data\raw | 존재: True
전처리 데이터 폴더: D:\dev\llm-data-analysis-course\data\processed | 존재: True
보고서 폴더: D:\dev\llm-data-analysis-course\reports | 존재: True


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [2]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


In [3]:
# [추가 검증] raw 원본 보존 확인용 해시 (마지막 Evidence 셀에서 다시 계산해 비교)
FILE_NAMES = ['customers', 'products', 'orders', 'order_items']


def md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


raw_hash_start = {name: md5(RAW_DIR / f'{name}.csv') for name in FILE_NAMES}
for name, h in raw_hash_start.items():
    print(f'{name:12s} {h}')

customers    039ba6fd09d987defaa215d97a7af86a
products     4febd3009adf3b4b664783dbc53af3b3
orders       a7b45a48d269a9d5c76832ca210f8d0e
order_items  a1f030f3c5ae0cb9c16123693337ca07


### STEP 3. 전처리 전 상태 기록 (가이드 Evidence)

`isna()` 결측뿐 아니라, 숫자·날짜여야 하는데 문자열로 읽힌 컬럼과 그 안의 "바로 변환되지 않는 값", 범주 컬럼의 실제 고유값 수까지 기록합니다.

In [4]:
# [추가 검증] STEP 3. 전처리 전 상태 기록
raw_data = {'customers': customers, 'products': products, 'orders': orders, 'order_items': order_items}
key_columns = {'customers': 'customer_id', 'products': 'product_id', 'orders': 'order_id', 'order_items': 'order_item_id'}

before_summary = pd.DataFrame([
    {
        'dataset': name,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'missing_cells': int(df.isna().sum().sum()),
        'dup_rows': int(df.duplicated().sum()),
        'key_column': key_columns[name],
        'key_dup': int(df[key_columns[name]].duplicated().sum()),
    }
    for name, df in raw_data.items()
])
print('[전처리 전 상태]')
display(before_summary)

print('[결측이 있는 컬럼]')
for name, df in raw_data.items():
    na = df.isna().sum()
    print(f'  {name}:', na[na > 0].to_dict() or '없음')


def non_numeric_values(series):
    converted = pd.to_numeric(series, errors='coerce')
    return series[converted.isna() & series.notna()].unique().tolist()


def non_date_values(series):
    converted = pd.to_datetime(series, errors='coerce')
    return series[converted.isna() & series.notna()].unique().tolist()


print('\n[타입 문제 후보] 숫자·날짜여야 하는 컬럼의 dtype과 바로 변환되지 않는 값')
for name, col, kind in [('customers', 'age', 'number'), ('products', 'price', 'number'),
                        ('order_items', 'quantity', 'number'), ('order_items', 'unit_price', 'number'),
                        ('customers', 'signup_date', 'date'), ('orders', 'order_date', 'date')]:
    s = raw_data[name][col]
    bad = non_numeric_values(s) if kind == 'number' else non_date_values(s)
    print(f'  {name + "." + col:24s} dtype={str(s.dtype):6s} 바로 변환 불가 {len(bad)}종: {bad}')

print('\n[범주 컬럼 실제 값]')
print('  gender:', customers['gender'].value_counts(dropna=False).to_dict())
print('  order_status 고유값', orders['order_status'].nunique(), '종 / payment_method', orders['payment_method'].nunique(),
      '종 / category', products['category'].nunique(), '종 / city', customers['city'].nunique(), '종')

[전처리 전 상태]


,dataset,rows,columns,missing_cells,dup_rows,key_column,key_dup
0,customers,152,6,6,2,customer_id,2
1,products,102,4,0,2,product_id,2
2,orders,302,5,0,2,order_id,2
3,order_items,765,5,0,2,order_item_id,2


[결측이 있는 컬럼]
  customers: {'age': 4, 'city': 2}
  products: 없음
  orders: 없음
  order_items: 없음

[타입 문제 후보] 숫자·날짜여야 하는 컬럼의 dtype과 바로 변환되지 않는 값
  customers.age            dtype=str    바로 변환 불가 1종: ['unknown']
  products.price           dtype=str    바로 변환 불가 5종: ['75,000', 'unknown', '181,000', '28,000', '93,000']
  order_items.quantity     dtype=str    바로 변환 불가 5종: ['3개', '2개', '5개', 'three', 'unknown']
  order_items.unit_price   dtype=str    바로 변환 불가 7종: ['52,000', '17,000', 'unknown', '161,000', 'not-a-number', '34,000', '129,000']
  customers.signup_date    dtype=str    바로 변환 불가 2종: ['2026-13-40', 'not-a-date']
  orders.order_date        dtype=str    바로 변환 불가 3종: ['not-a-date', '2026-02-30', '2026/99/99']

[범주 컬럼 실제 값]
  gender: {'M': 81, 'F': 71}
  order_status 고유값 15 종 / payment_method 9 종 / category 13 종 / city 15 종


## 답안 1. 전처리 전 상태 기록

- 각 데이터 shape: customers (152, 6), products (102, 4), orders (302, 5), order_items (765, 5)
- 결측치: customers만 6셀 (age 4, city 2). 나머지 3개 파일은 `isna()` 기준 0
- 전체 중복: 4개 파일 모두 2행씩 (총 8행)
- 주요 ID 중복: customer_id 31·76, product_id 30·84, order_id 44·205, order_item_id 109·549 (각 2건)
- 타입 문제 후보: age, price, quantity, unit_price, signup_date, order_date 6개 컬럼이 모두 문자열(`str`)로 읽혔습니다.
  - 숫자로 바로 바뀌지 않는 값: 쉼표 숫자(`75,000` 등), `unknown`, `not-a-number`, `3개`, `2개`, `5개`, `three`
  - 날짜로 바로 바뀌지 않는 값: `2026-13-40`, `not-a-date`, `2026-02-30`, `2026/99/99`
- 범주 고유값: order_status 15종, payment_method 9종, category 13종, city 15종 (정상이라면 각각 3, 4, 7, 10종)

![전처리 전 상태](images/step01_before.png)

- Evidence: 위 `[추가 검증] STEP 3` 셀 Output

### 결과 관찰
- `isna()`로 보면 결측은 customers의 6셀뿐입니다. 하지만 products, orders, order_items에도 계산에 쓸 수 없는 값이 문자열로 들어 있습니다. `unknown`, `not-a-date`처럼 결측을 글자로 적은 값입니다.
- 숫자 컬럼 4개가 문자열이라 이 상태로는 `describe()`나 `price <= 0` 같은 비교를 할 수 없습니다.
- 범주 고유값이 정상 개수보다 모두 많습니다. 표기 차이가 있다는 뜻입니다.

### 나의 해석과 판단
처리 순서는 다음과 같이 판단했습니다.
1. 타입 변환을 먼저 해야 합니다. 숫자로 바뀌기 전에는 결측도 이상값도 제대로 셀 수 없습니다. 지금의 "결측 6셀"은 문자열로 숨은 값을 빼고 센 과소 집계입니다.
2. 중복은 지우기 전에 "키 중복이 전체 행 중복과 같은 행인가"를 확인해야 합니다. 같으면 이중 적재로 보고 지워도 정보 손실이 없습니다. 다르면 어느 행이 맞는지 판단해야 하므로 자동 삭제하면 안 됩니다.
3. 범주 표준화와 결측·이상값 처리는 그다음입니다. 처리 건수를 셀 수 있는 상태에서 해야 처리 결과를 검증할 수 있습니다.

### 업무·분석적 의미
- "결측 0"이 "품질 정상"을 뜻하지 않습니다. 숫자 필드에 `3개`, `three`가 들어 있다는 것은 입력 화면이나 수집 시스템이 자유 텍스트를 허용한다는 신호입니다. 분석 쪽에서 고치는 것과 별개로 원천 시스템에 공유할 내용입니다.

### 한계와 추가 확인 사항
- `unknown`이 응답 거부인지 시스템 기본값인지는 데이터에 기록이 없어 구분할 수 없습니다.
- 중복 2행이 같은 배치에서 두 번 적재된 것인지는 적재 이력 컬럼이 없어 확인할 수 없습니다.

## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [5]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,152,6
1,products,102,4
2,orders,302,5
3,order_items,765,5


In [6]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [7]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [8]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [9]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
age,4,2.63
city,2,1.32
name,0,0.00
customer_id,0,0.00
gender,0,0.00
signup_date,0,0.00


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [10]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값:

 41.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


In [11]:
# [추가 검증] 결측 처리 건수와 영향 범위 (원본과 대조)
age_numeric_raw = pd.to_numeric(customers['age'], errors='coerce')
age_orig_missing = customers['age'].isna()
age_conv_failed = age_numeric_raw.isna() & customers['age'].notna()
imputed_ids = customers.loc[age_numeric_raw.isna(), 'customer_id'].drop_duplicates().tolist()
unknown_city_ids = customers.loc[customers['city'].isna(), 'customer_id'].drop_duplicates().tolist()

print(f'age 대체 건수: {int(age_numeric_raw.isna().sum())} (원래 결측 {int(age_orig_missing.sum())} + "unknown" 변환 실패 {int(age_conv_failed.sum())})')
print(f'age 평균 {age_numeric_raw.mean():.2f} / 중앙값 {age_median} (150세 포함 상태)')
print('age 대체 고객 customer_id:', imputed_ids, '→ 이 고객들의 원본 주문 수:', int(orders['customer_id'].isin(imputed_ids).sum()))
print('city Unknown 고객 customer_id:', unknown_city_ids, '→ 원본 주문 수:', int(orders['customer_id'].isin(unknown_city_ids).sum()))
print('대체 후 age == 41.0 인 고객 수:', int((customers_clean['age'] == age_median).sum()), '(원래 41세였던 고객 포함)')

age 대체 건수: 6 (원래 결측 4 + "unknown" 변환 실패 2)
age 평균 43.86 / 중앙값 41.0 (150세 포함 상태)
age 대체 고객 customer_id: [12, 24, 53, 88, 97, 121] → 이 고객들의 원본 주문 수: 9
city Unknown 고객 customer_id: [41, 89] → 원본 주문 수: 3
대체 후 age == 41.0 인 고객 수: 11 (원래 41세였던 고객 포함)


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [12]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [13]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


In [14]:
# [추가 검증] 키 중복이 전체 행 중복과 같은 행인지 확인 (다르면 자동 삭제 금지)
dup_rows = []
for name, df in zip(FILE_NAMES, [customers_clean, products_clean, orders_clean, order_items_clean]):
    key = key_columns[name]
    key_dup = df[key].duplicated()
    dup_rows.append({
        'dataset': name,
        'full_row_dup': int(df.duplicated().sum()),
        'key_dup': int(key_dup.sum()),
        'key_dup_not_full_dup': int((key_dup & ~df.duplicated()).sum()),
        'dup_key_values': sorted(int(v) for v in df.loc[key_dup, key]),
    })
duplicate_check = pd.DataFrame(dup_rows)
display(duplicate_check)

items_per_order = order_items_clean.groupby('order_id').size()
orders_per_customer = orders_clean.groupby('customer_id').size()
print('[관계용 ID의 자연스러운 반복]')
print('  order_items.order_id: 주문', items_per_order.size, '개, 주문당 상세 행 수 분포', items_per_order.value_counts().sort_index().to_dict())
print('  orders.customer_id: 고객', orders_per_customer.size, '명, 고객당 최대 주문', int(orders_per_customer.max()), '건')

,dataset,full_row_dup,key_dup,key_dup_not_full_dup,dup_key_values
0,customers,2,2,0,"[31, 76]"
1,products,2,2,0,"[30, 84]"
2,orders,2,2,0,"[44, 205]"
3,order_items,2,2,0,"[109, 549]"


[관계용 ID의 자연스러운 반복]
  order_items.order_id: 주문 302 개, 주문당 상세 행 수 분포 {1: 75, 2: 68, 3: 82, 4: 77}
  orders.customer_id: 고객 128 명, 고객당 최대 주문 7 건


In [15]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [16]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22096\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22096\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://p

In [17]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

order_status
completed    175
cancelled     58
refunded      52
Complete       2
cancel         2
Refund         2
complete       1
COMPLETED      1
완료             1
Cancel         1
CANCELLED      1
취소             1
refund         1
REFUNDED       1
환불             1
Name: count, dtype: int64

In [18]:
# [추가 검증] 문자열 정리 전후 비교 (원본 vs 정리 후)
def whitespace_count(series):
    s = series.dropna().astype(str)
    return int((s != s.str.strip()).sum())


strip_rows = []
for (name, raw_df, clean_df) in [('customers', customers, customers_clean), ('products', products, products_clean),
                                 ('orders', orders, orders_clean)]:
    for col in clean_df.select_dtypes(include=['object', 'str']).columns:
        if col in ('order_date', 'signup_date', 'price', 'age'):
            continue
        strip_rows.append({'column': f'{name}.{col}', 'whitespace_before': whitespace_count(raw_df[col]),
                           'whitespace_after': whitespace_count(clean_df[col]),
                           'nunique_before': raw_df[col].nunique(), 'nunique_after': clean_df[col].nunique(),
                           'empty_string_after': int((clean_df[col].dropna().astype(str) == '').sum())})
display(pd.DataFrame(strip_rows))
print('strip 후 customers 결측 셀:', int(customers_clean.isna().sum().sum()), "| 문자열 'nan' 생성:", int((customers_clean['city'] == 'nan').sum()))

,column,whitespace_before,whitespace_after,nunique_before,nunique_after,empty_string_after
0,customers.name,0,0,147,147,0
1,customers.gender,0,0,2,2,0
2,customers.city,5,0,15,11,0
3,products.product_name,0,0,100,100,0
4,products.category,3,0,13,10,0
5,orders.payment_method,5,0,9,4,0
6,orders.order_status,0,0,15,15,0


strip 후 customers 결측 셀: 0 | 문자열 'nan' 생성: 0


## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [19]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

In [20]:
# [추가 검증] 허용값 밖 상태값이 남았는지 확인 + strip으로 해결되지 않는 category 표기
ALLOWED_STATUS = {'completed', 'cancelled', 'refunded'}
print('표준화 전 order_status:', orders['order_status'].nunique(), '종 →', '표준화 후', orders_clean['order_status'].nunique(), '종')
unexpected = set(orders_clean['order_status'].dropna()) - ALLOWED_STATUS
print('허용값 밖 상태값:', unexpected or '없음', '| 합계', len(orders_clean))
assert not unexpected

print('\ncategory (strip 후):', products_clean['category'].value_counts().to_dict())
category_nospace = products_clean['category'].str.replace(' ', '', regex=False)
print('글자 사이 공백 제거 시 고유값:', products_clean['category'].nunique(), '→', category_nospace.nunique())

name_prefix = products_clean['product_name'].str.split(' 상품').str[0]
price_preview = pd.to_numeric(products_clean['price'].astype(str).str.replace(',', ''), errors='coerce')  # 비교용 (31번 셀 전)
cat_median = price_preview[price_preview.between(1, 1_000_000)].groupby(category_nospace).median()
mismatch = products_clean.loc[name_prefix != category_nospace, ['product_id', 'product_name', 'category', 'price']].copy()
mismatch['name_category_median'] = name_prefix[mismatch.index].map(cat_median)
mismatch['category_median'] = category_nospace[mismatch.index].map(cat_median)
print('상품명 접두어와 category가 다른 상품:', len(mismatch), '개 (가격 비교는 참고용)')
display(mismatch)

표준화 전 order_status: 15 종 → 표준화 후 3 종
허용값 밖 상태값: 없음 | 합계 300

category (strip 후): {'패션': 17, '뷰티': 17, '식품': 16, '도서': 14, '전자기기': 13, '스포츠': 11, '생활용품': 9, '전자 기기': 1, '생활 용품': 1, '뷰 티': 1}
글자 사이 공백 제거 시 고유값: 10 → 7
상품명 접두어와 category가 다른 상품: 5 개 (가격 비교는 참고용)


,product_id,product_name,category,price,name_category_median,category_median
2,3,뷰티 상품 003,전자기기,48000,64500.0,127000.0
26,27,뷰티 상품 027,전자 기기,21000,64500.0,127000.0
43,44,스포츠 상품 044,생활용품,29000,64500.0,75500.0
69,70,도서 상품 070,패션,168000,96000.0,103500.0
77,78,스포츠 상품 078,뷰 티,54000,64500.0,64500.0


## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [21]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-12,2026-07,Sunday
1,2025-10-02,2025-10,Thursday
2,2026-03-13,2026-03,Friday
3,2025-10-12,2025-10,Sunday
4,2026-04-17,2026-04,Friday


In [22]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 2


,customer_id,signup_date
0,1,2025-10-05
1,2,2023-09-20
2,3,2025-01-22
3,4,2025-03-13
4,5,2024-02-10


In [23]:
# [추가 검증] 날짜 변환 결과를 실제 값으로 확인: 실패 행, 유효 범위, 미래 날짜, 가입일과의 순서
AS_OF = pd.Timestamp('2026-10-22')   # 작성일
nat_orders = orders_clean.loc[orders_clean['order_date'].isna(), ['order_id', 'order_status']]
nat_orders['raw_order_date'] = orders.loc[nat_orders.index, 'order_date']
print('order_date 변환 실패 주문:', nat_orders.to_dict('records'))

future = orders_clean.loc[orders_clean['order_date'] > AS_OF, ['order_id', 'order_date', 'order_status']]
print('작성일(2026-10-22) 이후 주문:', [(int(r.order_id), r.order_date.strftime('%Y-%m-%d'), r.order_status) for r in future.itertuples()])

valid_dates = orders_clean.loc[orders_clean['order_date'] <= AS_OF, 'order_date']
VALID_START, VALID_END = valid_dates.min(), valid_dates.max()
print('유효 주문일 범위:', VALID_START.date(), '~', VALID_END.date())
print('월별 주문 수:', orders_clean['order_date'].dt.to_period('M').astype(str).value_counts(dropna=False).sort_index().to_dict())

signup = customers_clean.drop_duplicates('customer_id').set_index('customer_id')['signup_date']
order_signup = orders_clean.assign(signup_date=orders_clean['customer_id'].map(signup))
comparable = order_signup.dropna(subset=['order_date', 'signup_date'])
before_signup = comparable[comparable['order_date'] < comparable['signup_date']]
print(f'\n주문일 < 가입일 주문: {len(before_signup)}건 / 비교 가능 {len(comparable)}건 ({len(before_signup) / len(comparable) * 100:.1f}%)')
print('  상태별:', before_signup['order_status'].value_counts().to_dict(), '| 고객 수:', before_signup['customer_id'].nunique())
print('  가입일보다 최대', int((before_signup['signup_date'] - before_signup['order_date']).dt.days.max()), '일 앞선 주문 존재')

order_date 변환 실패 주문: [{'order_id': 72, 'order_status': 'completed', 'raw_order_date': 'not-a-date'}, {'order_id': 143, 'order_status': 'refunded', 'raw_order_date': '2026-02-30'}, {'order_id': 214, 'order_status': 'completed', 'raw_order_date': '2026/99/99'}]
작성일(2026-10-22) 이후 주문: [(256, '2027-01-15', 'completed'), (287, '2028-12-31', 'cancelled')]
유효 주문일 범위: 2025-07-01 ~ 2026-08-30
월별 주문 수: {'2025-07': 19, '2025-08': 26, '2025-09': 12, '2025-10': 22, '2025-11': 23, '2025-12': 23, '2026-01': 26, '2026-02': 19, '2026-03': 22, '2026-04': 26, '2026-05': 20, '2026-06': 24, '2026-07': 12, '2026-08': 21, '2027-01': 1, '2028-12': 1, nan: 3}



주문일 < 가입일 주문: 58건 / 비교 가능 289건 (20.1%)
  상태별: {'completed': 33, 'cancelled': 14, 'refunded': 11} | 고객 수: 32
  가입일보다 최대 308 일 앞선 주문 존재


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [24]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 2
quantity 변환 실패: 5
unit_price 변환 실패: 2


In [25]:
# [추가 검증] 변환 실패를 '원래 결측'과 분리해 원본 값까지 기록 (index로 원본과 정렬)
def conversion_report(dataset, column, before, after):
    failed = after.isna() & before.notna()
    comma = before.astype(str).str.contains(',', regex=False) & after.notna()
    return {'dataset': dataset, 'column': column, 'orig_missing': int(before.isna().sum()),
            'conversion_failed': int(failed.sum()), 'failed_values': before[failed].unique().tolist(),
            'comma_recovered': int(comma.sum())}


conversion_summary = pd.DataFrame([
    conversion_report('customers', 'age', customers['age'], age_numeric_raw),
    conversion_report('customers', 'signup_date', customers.loc[customers_clean.index, 'signup_date'], customers_clean['signup_date']),
    conversion_report('products', 'price', products.loc[products_clean.index, 'price'], products_clean['price']),
    conversion_report('orders', 'order_date', orders.loc[orders_clean.index, 'order_date'], orders_clean['order_date']),
    conversion_report('order_items', 'quantity', order_items.loc[order_items_clean.index, 'quantity'], order_items_clean['quantity']),
    conversion_report('order_items', 'unit_price', order_items.loc[order_items_clean.index, 'unit_price'], order_items_clean['unit_price']),
])
print('[숫자·날짜 변환 결과]')
display(conversion_summary)
print('변환 실패 합계:', int(conversion_summary['conversion_failed'].sum()), '건')
print('dtype:', {c: str(d) for c, d in [('age', customers_clean['age'].dtype), ('price', products_clean['price'].dtype),
                                        ('quantity', order_items_clean['quantity'].dtype), ('order_date', orders_clean['order_date'].dtype)]})
print('order_date 범위(변환 성공분):', orders_clean['order_date'].min().date(), '~', orders_clean['order_date'].max().date())
print('signup_date 범위(변환 성공분):', customers_clean['signup_date'].min().date(), '~', customers_clean['signup_date'].max().date())

[숫자·날짜 변환 결과]


,dataset,column,orig_missing,conversion_failed,failed_values,comma_recovered
0,customers,age,4,2,[unknown],0
1,customers,signup_date,0,2,"[2026-13-40, not-a-date]",0
2,products,price,0,2,[unknown],4
3,orders,order_date,0,3,"[not-a-date, 2026-02-30, 2026/99/99]",0
4,order_items,quantity,0,5,"[3개, 2개, 5개, three, unknown]",0
5,order_items,unit_price,0,2,"[unknown, not-a-number]",5


변환 실패 합계: 16 건
dtype: {'age': 'float64', 'price': 'float64', 'quantity': 'float64', 'order_date': 'datetime64[us]'}
order_date 범위(변환 성공분): 2025-07-01 ~ 2028-12-31
signup_date 범위(변환 성공분): 2023-07-21 ~ 2026-08-24


## 답안 2. 문자열·타입·결측 처리

- 적용한 문자열 정리: 모든 문자열 컬럼 앞뒤 공백 제거. 공백이 있던 값은 city 5, category 3, payment_method 5개입니다. 고유값 수는 city 15 → 11 (10개 도시 + `Unknown`), category 13 → 10, payment_method 9 → 4로 바뀌었습니다. 빈 문자열 0건, 문자열 `'nan'` 생성 0건, strip 전후 결측 수 변화 없음
- 숫자 변환 대상과 실패 건수: age 2 (`unknown`), price 2 (`unknown`, 쉼표 4건은 복구), quantity 5 (`3개`, `2개`, `5개`, `three`, `unknown`), unit_price 2 (`unknown`, `not-a-number`, 쉼표 5건은 복구)
- 날짜 변환 대상과 실패 건수: signup_date 2 (`2026-13-40`, `not-a-date`), order_date 3 (`not-a-date`, `2026-02-30`, `2026/99/99` → 주문 72 completed, 143 refunded, 214 completed). 변환 성공분 범위는 order_date 2025-07-01 ~ 2028-12-31, signup_date 2023-07-21 ~ 2026-08-24
- 결측 처리 기준: age 6건(원래 결측 4 + 변환 실패 2, customer_id 12·24·53·88·97·121)은 중앙값 41.0으로 대체했습니다. city 2건(customer_id 41·89)은 `Unknown`으로 처리했습니다. 날짜 변환 실패는 대체하지 않고 `NaT`로 남겼습니다.

![타입과 결측 처리](images/step02_cleaning.png)

- Evidence: 위 8번 뒤 `[추가 검증]`, 10번 뒤 `[추가 검증]`, 13번 뒤 `[추가 검증]` 셀 Output

### 결과 관찰
- 변환 실패는 모두 16건입니다(customers 4, products 2, orders 3, order_items 7).
- `3개`, `2개`, `5개`는 사람이 보면 의미가 분명한데 `three`, `unknown`과 똑같이 `NaN`이 되었습니다.
- age 평균은 43.86, 중앙값은 41.0입니다. 대체 후 41세 고객은 5명에서 11명이 되었습니다.
- 10번 문자열 정리 셀에서 `Pandas4Warning`이 출력됐습니다. 공식 함수가 `select_dtypes(include='object')`로 문자열 컬럼을 고르는데, pandas 3에서는 하위 호환 때문에 `str` 컬럼도 함께 선택되고 있습니다. 이 동작은 앞으로 제거될 예정이라는 경고입니다.

### 나의 해석과 판단
- age 중앙값 대체: 150세 같은 극단값에 평균보다 덜 민감합니다. 대체 대상도 6명(150명의 4%)이라 분포 영향이 작습니다. 다만 40대 구간 고객이 24명 → 30명으로 늘어납니다(STEP 17 검증 셀). 어떤 값이 실제이고 어떤 값이 대체값인지 표시하는 컬럼이 없다는 것이 정보 손실입니다.
- city `Unknown`: 도시는 평균 개념이 없습니다. 최빈값으로 채우면 특정 도시 고객 수를 부풀립니다. `Unknown`으로 두면 지역 분석에서 "모르는 고객"을 따로 볼 수 있습니다. 이 2명의 주문은 3건으로 작습니다.
- 날짜 미대체: 주문일은 대표값이 존재하지 않는 값입니다. 임의 날짜를 넣으면 월별 집계가 바뀝니다. 그래서 `NaT`로 남기고 분석 기간에서 제외하는 것이 맞다고 판단했습니다. 이 3건 중 2건이 completed 주문이라 월별 completed 상품 주문 금액에서 빠진다는 점은 기록해 둡니다.
- 경고 해석: 지금은 결과가 맞지만, pandas 버전이 올라가면 같은 코드가 오류 없이 아무 컬럼도 strip하지 않을 수 있습니다. 문자열 정리 후 고유값 수를 다시 세는 검증(이번 `[추가 검증]`)을 남겨야 이런 변화를 잡을 수 있습니다.

### 업무·분석적 의미
- `errors="coerce"`는 오류를 해결하는 옵션이 아니라 결측으로 바꿔 보이지 않게 하는 옵션입니다. 실패 건수와 실패한 원본 값 목록을 남기면 입력 시스템의 어느 필드를 고쳐야 하는지도 알 수 있습니다.

### 한계와 추가 확인 사항
- age 결측 원인(미입력, 응답 거부)은 데이터에 없어 대체 방식이 원인에 맞는지 확인할 수 없습니다.
- 공식 순서는 중복 제거 전 152행으로 중앙값을 계산합니다. 중복 제거 후, 150세 제외 후 다시 계산해도 41.0으로 같았습니다(STEP 17 검증 셀). 기준 선택이 결과를 바꾸지 않았습니다.

## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [26]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06
Name: price, dtype: float64

,quantity,unit_price
count,758.000000,761.000000
mean,3.168865,103494.086728
std,4.159055,56969.823769
min,-1.000000,0.000000
25%,2.000000,55000.000000
50%,3.000000,96000.000000
75%,4.000000,154000.000000
max,100.000000,197000.000000


In [27]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


In [28]:
# [추가 검증] 이상값·변환 실패 후보를 주문 상태와 대조 (실습 기준 적용 전)
status_lookup = orders_clean.set_index('order_id')['order_status']
oi = order_items_clean
cand_mask = (oi['quantity'].isna() | (oi['quantity'] <= 0) | (oi['quantity'] > 10)
             | oi['unit_price'].isna() | (oi['unit_price'] <= 0))
candidates = oi.loc[cand_mask, ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']].copy()
candidates.insert(3, 'raw_quantity', order_items.loc[candidates.index, 'quantity'])
candidates.insert(5, 'raw_unit_price', order_items.loc[candidates.index, 'unit_price'])
candidates['order_status'] = candidates['order_id'].map(status_lookup)
candidates['실습 기준'] = np.where((candidates['quantity'] > 0) & (candidates['unit_price'] > 0), '유지', '제외')
candidates['사유'] = np.select(
    [candidates['quantity'].isna(), candidates['unit_price'].isna(), candidates['quantity'] <= 0,
     candidates['unit_price'] <= 0],
    ['quantity 변환 실패', 'unit_price 변환 실패', 'quantity 0 이하', 'unit_price 0 이하'], default='quantity > 10')
print('[order_items 후보 13행]')
display(candidates)
print('제외 대상 중 completed 주문 행:', int(((candidates['실습 기준'] == '제외') & (candidates['order_status'] == 'completed')).sum()))

# 상품 마스터 가격 이상 vs 실제 주문 단가
bad_master = products_clean.loc[products_clean['price'].isna() | (products_clean['price'] <= 0) | (products_clean['price'] > 1_000_000),
                                ['product_id', 'category', 'price']].copy()
bad_master.insert(2, 'raw_price', products.loc[bad_master.index, 'price'])
unit_stats = oi.groupby('product_id')['unit_price'].agg(order_rows='size', unit_price_nunique='nunique', unit_price_min='min', unit_price_max='max')
bad_master = bad_master.merge(unit_stats, on='product_id', how='left')
print('\n[상품 마스터 가격 후보 vs 주문 상세 unit_price]')
display(bad_master)

valid_master = products_clean.loc[products_clean['price'] > 0].query('price <= 1_000_000').set_index('product_id')['price']
cmp = oi.dropna(subset=['unit_price'])
cmp = cmp[cmp['product_id'].isin(valid_master.index) & (cmp['unit_price'] > 0)]
print(f'정상 마스터 가격 상품의 주문 상세 {len(cmp)}행 중 unit_price == products.price: {int((cmp["unit_price"] == cmp["product_id"].map(valid_master)).sum())}행')
print('customer_id 33 (age 150) 원본 주문 수:', int((orders['customer_id'] == 33).sum()))

[order_items 후보 13행]


,order_item_id,order_id,product_id,raw_quantity,quantity,raw_unit_price,unit_price,order_status,실습 기준,사유
16,17,8,76,3개,NaN,56000,56000.0,cancelled,제외,quantity 변환 실패
90,91,42,65,2개,NaN,193000,193000.0,completed,제외,quantity 변환 실패
204,205,84,41,5개,NaN,119000,119000.0,cancelled,제외,quantity 변환 실패
276,277,111,30,5,5.0,unknown,NaN,completed,제외,unit_price 변환 실패
318,319,129,57,three,NaN,96000,96000.0,completed,제외,quantity 변환 실패
432,433,171,87,5,5.0,not-a-number,NaN,refunded,제외,unit_price 변환 실패
446,447,175,79,unknown,NaN,153000,153000.0,completed,제외,quantity 변환 실패
510,511,201,17,0,0.0,76000,76000.0,refunded,제외,quantity 0 이하
578,579,229,17,2,2.0,0,0.0,cancelled,제외,unit_price 0 이하
611,612,241,24,-1,-1.0,42000,42000.0,refunded,제외,quantity 0 이하


제외 대상 중 completed 주문 행: 4

[상품 마스터 가격 후보 vs 주문 상세 unit_price]


,product_id,category,raw_price,price,order_rows,unit_price_nunique,unit_price_min,unit_price_max
0,18,패션,unknown,NaN,5,1,140000.0,140000.0
1,48,뷰티,0,0.0,9,1,25000.0,25000.0
2,73,도서,unknown,NaN,6,1,156000.0,156000.0
3,95,스포츠,-20000,-20000.0,4,1,82000.0,82000.0
4,99,뷰티,9999999,9999999.0,8,1,187000.0,187000.0


정상 마스터 가격 상품의 주문 상세 726행 중 unit_price == products.price: 726행
customer_id 33 (age 150) 원본 주문 수: 2


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [29]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (96, 4)
이상값 처리 후 order_items: (752, 5)


In [30]:
# [추가 검증] 위 셀의 '> 0' 필터가 지운 행을 원인별로 분리
removed_items = candidates[candidates['실습 기준'] == '제외']
print('order_items 제외 행 원인별:', removed_items['사유'].value_counts().to_dict(), '/ 합계', len(removed_items))
removed_products = bad_master[~bad_master['product_id'].isin(products_clean['product_id'])]
print('products 제외 상품:', removed_products[['product_id', 'raw_price']].to_dict('records'))
print('NaN > 0 결과:', bool((pd.Series([np.nan]) > 0).iloc[0]), '→ 변환 실패 행도 0 이하 기준에 섞여 제외됨')
print('유지된 이상값 후보: age 150, price 9,999,999 (product 99), quantity 50·100')

order_items 제외 행 원인별: {'quantity 변환 실패': 5, 'quantity 0 이하': 3, 'unit_price 변환 실패': 2, 'unit_price 0 이하': 1} / 합계 11
products 제외 상품: [{'product_id': 18, 'raw_price': 'unknown'}, {'product_id': 48, 'raw_price': '0'}, {'product_id': 73, 'raw_price': 'unknown'}, {'product_id': 95, 'raw_price': '-20000'}]
NaN > 0 결과: False → 변환 실패 행도 0 이하 기준에 섞여 제외됨
유지된 이상값 후보: age 150, price 9,999,999 (product 99), quantity 50·100


## 답안 3. 중복·범주 표준화·이상값 후보

- 제거/보류한 중복: 4개 파일에서 2행씩 모두 제거했습니다. 키 중복 8건이 모두 전체 행 중복과 같은 행이었습니다(`key_dup_not_full_dup` = 0). 보류한 중복은 없습니다. `order_items.order_id` 반복(주문당 1~4행)과 `orders.customer_id` 반복(고객 128명, 최대 7건)은 1:N 관계라 그대로 두었습니다.
- 표준화한 범주값: order_status 15종 → 3종 (completed 180, cancelled 63, refunded 57, 합계 300). payment_method는 공백 제거로 9종 → 4종
- 허용값 밖 값: order_status는 없습니다(`assert` 통과). category에는 글자 사이 공백 변형 `전자 기기`, `생활 용품`, `뷰 티`가 남았습니다(10종 = 정상 7 + 변형 3). 상품명 접두어와 category가 다른 상품도 5개 있습니다(product_id 3, 27, 44, 70, 78).
- 이상값 후보와 주문 상태 대조 (14번 뒤 `[추가 검증]`):

| 후보 | 행 | 주문 상태 | 실습 기준 |
| --- | --- | --- | --- |
| quantity 변환 실패 (`3개`, `2개`, `5개`, `three`, `unknown`) | 5 | completed 3, cancelled 2 | 제외 |
| unit_price 변환 실패 (`unknown`, `not-a-number`) | 2 | completed 1, refunded 1 | 제외 |
| quantity 0, 0, -1 | 3 | refunded 2 (-1 포함), cancelled 1 | 제외 |
| unit_price 0 | 1 | cancelled | 제외 |
| quantity 50 / 100 | 2 | completed / refunded | 유지 |
| products.price `unknown`·`unknown`·0·-20,000 (상품 18, 73, 48, 95) | 4 | 주문 상세 24행 | 상품 제외 |
| products.price 9,999,999 (상품 99), customers.age 150 (고객 33) | 2 | 주문 상세 8행 / 주문 2건 | 유지 |

- 실습 기준 적용 결과: products 100 → 96행, order_items 763 → 752행. 실제로 0 이하인 값은 6건뿐이었습니다. 나머지 9건(price 2, quantity 5, unit_price 2)은 변환 실패 `NaN`이 `> 0` 비교에서 False가 되어 같이 제외됐습니다.

![중복 범주 이상값](images/step03_rules.png)

- Evidence: 9·11·14번 뒤 `[추가 검증]` 셀과 16번 앞 `[추가 검증]` 셀 Output

### 결과 관찰
- 상품 마스터 가격이 이상한 5개 상품(18, 48, 73, 95, 99)은 주문 상세 unit_price가 상품마다 한 가지 값으로 일정했습니다(예: 상품 48은 9행 모두 25,000, 상품 99는 8행 모두 187,000).
- 마스터 가격이 정상인 상품의 주문 상세 726행은 모두 `unit_price == products.price`였습니다.
- quantity -1 행은 refunded 주문(241)에, quantity 100 행은 refunded 주문(295)에 있습니다.
- 제외된 11행 중 4행은 completed 주문의 행입니다. 그중 주문 175는 상세 행이 하나뿐이라, 제외 후 상세 행이 없는 completed 주문이 되었습니다(16번 뒤 `[추가 검증]`).

### 나의 해석과 판단
이상값을 자동 삭제하지 않고 후보로 검토해야 하는 이유는, 값의 의미가 값 자체가 아니라 다른 테이블과 업무 규칙에 있기 때문입니다. 이번에 대조해 보니 같은 "이상값"이라도 근거가 달랐습니다.
- 상품 마스터 가격: 정상 상품은 726행 모두 단가와 마스터 가격이 같은데, 이 5개 상품만 다릅니다. 주문 단가는 상품마다 일정하므로 마스터 가격이 잘못 입력된 것으로 판단했습니다. 그런데 실습 기준은 마스터 가격이 이상한 상품 4개를 지워서 단가가 정상인 주문 상세 24행의 상품 연결을 끊었습니다. 반대로 9,999,999는 기준에 걸리지 않아 남았습니다. 자동 기준이 지워야 할 것은 남기고, 남겨야 할 것을 지운 사례입니다.
- quantity -1: refunded 주문에만 있어 반품 표기일 가능성이 있습니다. 하나뿐이라 "반품은 음수로 기록한다"는 규칙이라고 단정할 수는 없습니다.
- quantity 100: refunded 주문이라 completed 기준 상품 주문 금액에는 영향이 없습니다. 반면 quantity 50은 completed 주문이고 4,800,000으로 completed 금액에 그대로 들어갑니다. 같은 "대량 수량" 후보라도 분석 영향이 다릅니다.
- 가장 문제라고 본 것은 `> 0` 필터 하나가 변환 실패, 0, 음수라는 성격이 다른 문제를 한꺼번에 지운다는 점입니다. 공식 요약 보고서에도 "0 이하 제외"만 적혀 있어서, 변환 실패 9행이 빠졌다는 사실이 남지 않습니다.

### 업무·분석적 의미
- category 표기 변형이 남으면 카테고리 집계에서 `뷰 티`가 별도 행으로 나오고 `뷰티` 합계는 줄어듭니다. 표준화는 보기 좋게 만드는 일이 아니라 집계 단위를 맞추는 일입니다.
- 상품명과 category가 다른 5개 상품은 가격을 카테고리 중앙값과 비교해 보면 5개 중 3개(3, 27, 44)가 상품명 쪽 카테고리에 더 가깝습니다. 하지만 이것은 참고 근거일 뿐이라 수정하지 않고 담당자 확인 항목으로 남겼습니다.

### 한계와 추가 확인 사항
- 반품 표기 규칙, 대량 주문 허용 범위, 0원 상품 정책은 데이터에 없습니다.
- category 글자 사이 공백은 공식 전처리가 처리하지 않아 저장 결과에도 남아 있습니다(25번 뒤 제안 기준에서 영향 비교).

## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [31]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [32]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계:

 252140000.0


In [33]:
# [추가 검증] line_total 재계산·상태별 구분 (상품 주문 금액이며 회계상 매출이 아님)
recalc = order_items_clean['quantity'] * order_items_clean['unit_price']
print('quantity × unit_price 재계산 일치:', bool((recalc == order_items_clean['line_total']).all()),
      '| 결측', int(order_items_clean['line_total'].isna().sum()), '| 0 이하', int((order_items_clean['line_total'] <= 0).sum()))
by_status = (order_items_clean.assign(order_status=order_items_clean['order_id'].map(status_lookup).fillna('(orders에 없음)'))
             .groupby('order_status')['line_total'].agg(rows='size', amount='sum'))
by_status['ratio(%)'] = (by_status['amount'] / by_status['amount'].sum() * 100).round(2)
display(by_status)
top = order_items_clean.nlargest(2, 'line_total')[['order_item_id', 'order_id', 'quantity', 'unit_price', 'line_total']]
top['order_status'] = top['order_id'].map(status_lookup)
display(top)
print('order_month 결측:', int(orders_clean['order_month'].isna().sum()), '| 문자열 "NaT":', int((orders_clean['order_month'] == 'NaT').sum()))

quantity × unit_price 재계산 일치: True | 결측 0 | 0 이하 0


,rows,amount,ratio(%)
order_status,,,
(orders에 없음),2,922000.0,0.37
cancelled,160,53805000.0,21.34
completed,446,141340000.0,56.06
refunded,144,56073000.0,22.24


,order_item_id,order_id,quantity,unit_price,line_total,order_status
750,751,295,100.0,155000.0,15500000.0,refunded
737,738,290,50.0,96000.0,4800000.0,completed


order_month 결측: 3 | 문자열 "NaT": 0


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [34]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.customer_id,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.product_id,26


In [35]:
# [추가 검증] invalid 원인 분리: 원본에도 없던 ID인가, 전처리로 생긴 고아인가?
print('없는 customer_id:', sorted(int(x) for x in invalid_customers['customer_id']),
      '| 원본 customers에 존재:', bool(customers['customer_id'].isin(invalid_customers['customer_id']).any()),
      '| 해당 주문:', invalid_customers[['order_id', 'order_status']].to_dict('records'))
print('없는 order_id:', sorted(int(x) for x in invalid_orders['order_id']),
      '| 원본 orders에 존재:', bool(orders['order_id'].isin(invalid_orders['order_id']).any()))

raw_product_ids = set(products['product_id'])
product_cause = (invalid_products.groupby('product_id').size().rename('order_item_rows').reset_index())
product_cause['raw_price'] = product_cause['product_id'].map(products.drop_duplicates().set_index('product_id')['price'])
product_cause['cause'] = np.where(product_cause['product_id'].isin(raw_product_ids), '전처리에서 상품 제외', '원본에도 없는 ID')
display(product_cause)
print(product_cause.groupby('cause')['order_item_rows'].sum().to_dict())

raw_fk = int((~order_items['product_id'].isin(products['product_id'])).sum())
print(f'product_id FK 문제: 원본 {raw_fk}행 → 전처리 후 {len(invalid_products)}행')

orders_without_items = orders_clean[~orders_clean['order_id'].isin(order_items_clean['order_id'])]
print('상세 행이 하나도 남지 않은 주문:', orders_without_items[['order_id', 'order_status']].to_dict('records'),
      '| 원본 기준:', int((~orders['order_id'].isin(order_items['order_id'])).sum()), '건')

없는 customer_id: [9991, 9992] | 원본 customers에 존재: False | 해당 주문: [{'order_id': 126, 'order_status': 'completed'}, {'order_id': 278, 'order_status': 'cancelled'}]
없는 order_id: [9991, 9992] | 원본 orders에 존재: False


,product_id,order_item_rows,raw_price,cause
0,18,5,unknown,전처리에서 상품 제외
1,48,9,0,전처리에서 상품 제외
2,73,6,unknown,전처리에서 상품 제외
3,95,4,-20000,전처리에서 상품 제외
4,9991,1,NaN,원본에도 없는 ID
5,9992,1,NaN,원본에도 없는 ID


{'원본에도 없는 ID': 2, '전처리에서 상품 제외': 24}
product_id FK 문제: 원본 2행 → 전처리 후 26행
상세 행이 하나도 남지 않은 주문: [{'order_id': 175, 'order_status': 'completed'}] | 원본 기준: 0 건


In [36]:
# [추가 검증] Chapter 06 분석 범위를 날짜까지 구체화: completed 주문, 유효 주문일, 관계가 온전한 행
scope = order_items_clean.merge(orders_clean[['order_id', 'customer_id', 'order_date', 'order_status']],
                                on='order_id', how='left', validate='many_to_one', indicator=True)
scope_steps = [('전처리 후 order_items', scope)]
scope = scope[scope['_merge'] == 'both']; scope_steps.append(('orders와 연결됨', scope))
scope = scope[scope['order_status'] == 'completed']; scope_steps.append(('completed', scope))
scope = scope[scope['order_date'].between(VALID_START, VALID_END)]; scope_steps.append((f'주문일 {VALID_START.date()}~{VALID_END.date()}', scope))
scope = scope[scope['product_id'].isin(products_clean['product_id'])]; scope_steps.append(('products와 연결됨(category 있음)', scope))
scope = scope[scope['customer_id'].isin(customers_clean['customer_id'])]; scope_steps.append(('(고객 분석 시에만) customers와 연결됨', scope))
display(pd.DataFrame([{'단계': s, '행': len(d), '주문': d['order_id'].nunique(), '상품 주문 금액': d['line_total'].sum()}
                      for s, d in scope_steps]))

,단계,행,주문,상품 주문 금액
0,전처리 후 order_items,752,301,252140000.0
1,orders와 연결됨,750,299,251218000.0
2,completed,446,179,141340000.0
3,주문일 2025-07-01~2026-08-30,437,176,138528000.0
4,products와 연결됨(category 있음),421,176,134513000.0
5,(고객 분석 시에만) customers와 연결됨,420,175,134137000.0


## 답안 4. PK/FK와 파생 컬럼 재검증

- 없는 `customer_id`: 2건 (9991, 9992). 원본 customers에도 없던 ID이고, 해당 주문은 126(completed), 278(cancelled)입니다.
- 없는 `order_id`: 2건 (9991, 9992). 원본 orders에도 없던 ID입니다.
- 없는 `product_id`: 26행 = 원본에도 없던 ID(9991, 9992) 2행 + 전처리에서 제외한 상품(18, 48, 73, 95)을 참조하는 24행. 원본 기준 2행에서 전처리 후 26행으로 늘었습니다.
- 상세 행이 없는 주문: 원본 0건 → 전처리 후 1건 (주문 175, completed)
- 생성한 파생 컬럼: `order_month`, `order_dayofweek` (orders), `line_total` (order_items)
- `line_total` 검증: quantity × unit_price 재계산 일치, 결측 0, 0 이하 0. 전체 합계 252,140,000을 주문 상태별로 나누면 completed 141,340,000 (56.06%), refunded 56,073,000 (22.24%), cancelled 53,805,000 (21.34%), orders에 없는 주문 922,000 (0.37%)입니다.
- 날짜 파생 컬럼: `order_month` 결측 3건(변환 실패 3건과 같음). 작성일(2026-10-22) 이후 주문 2건(256: 2027-01-15 completed, 287: 2028-12-31 cancelled). 주문일이 고객 가입일보다 앞선 주문 58건(비교 가능 289건 중 20.1%, 고객 32명, 최대 308일 앞섬)
- Chapter 06 분석 범위 (16번 뒤 두 번째 `[추가 검증]`):

| 단계 | 주문 상세 행 | 주문 | 상품 주문 금액 |
| --- | ---: | ---: | ---: |
| 전처리 후 order_items | 752 | 301 | 252,140,000 |
| orders와 연결됨 | 750 | 299 | 251,218,000 |
| completed | 446 | 179 | 141,340,000 |
| 주문일 2025-07-01 ~ 2026-08-30 | 437 | 176 | 138,528,000 |
| products와 연결됨 (category 있음) | 421 | 176 | 134,513,000 |
| (고객 분석 시에만) customers와 연결됨 | 420 | 175 | 134,137,000 |

![관계와 파생 컬럼 검증](images/step04_validation.png)

- Evidence: 12번 뒤, 15번 뒤, 16번 뒤 `[추가 검증]` 셀 Output

### 결과 관찰
- 전처리 전 product_id 연결 문제는 2행이었는데, 전처리 후 26행이 되었습니다. 늘어난 24행은 모두 quantity와 unit_price가 정상인 주문 상세입니다.
- `scripts/preprocess_data.py`는 invalid_count가 합계 30행이어도 "전처리 완료"를 출력하고 정상 종료합니다(22번 뒤 STEP 14 셀).
- completed 상품 주문 금액은 연결 조건을 하나씩 걸 때마다 줄어듭니다. 기간 조건에서 2,812,000, 상품 연결 조건에서 4,015,000이 빠집니다.

### 나의 해석과 판단
- 전처리 후 관계를 다시 검증해야 하는 이유는 전처리를 테이블별로 하지만 관계는 테이블 사이에 있기 때문입니다. products만 보면 "이상한 가격 4개 제외"는 합리적으로 보입니다. 하지만 그 결과로 정상 주문 상세 24행이 상품을 잃었고, 원본만 검사했다면 이 문제는 보이지 않았습니다.
- 이 상태로 Chapter 06에서 `inner` merge를 하면 24행이 조용히 빠집니다. `left` merge를 하면 category가 NaN이 되어 `groupby` 기본값(`dropna=True`)에서 빠집니다. 어느 쪽이든 오류 없이 카테고리별 금액만 줄어듭니다. 위 표에서 completed 금액 4,015,000이 category 연결 단계에서 빠지는 것이 그 크기입니다.
- 9991/9992는 원본부터 연결이 없던 ID입니다. 고객이 없는 주문 126(completed, 상세 1행)과 278(cancelled, 상세 3행), 주문이 없는 상세 2행, 상품이 없는 상세 2행이 있습니다. 삭제하기보다 "관계 불일치"로 따로 표시하고 분석에서 제외했다고 기록하는 것이 안전합니다.
- 주문일이 가입일보다 앞선 58건은 이번 전처리 기준으로 고칠 수 없는 문제입니다. 날짜 형식은 정상이라 변환 검증에는 걸리지 않습니다. `signup_date`의 의미(최초 가입일인지, 재가입일·회원 전환일인지)를 모르면 "신규 고객 주문" 같은 분석에 쓸 수 없다고 판단했습니다.

### 업무·분석적 의미
- 전처리가 데이터를 "더 깨끗하게" 만드는 것이 아니라 관계를 더 나쁘게 만들 수도 있다는 것을 숫자로 확인했습니다(FK 문제 2행 → 26행, 상세 없는 주문 0 → 1건).
- Chapter 06의 "completed 월별·카테고리별 상품 주문 금액"은 2025-07-01 ~ 2026-08-30, completed 176건, 421행, 134,513,000을 기준 합계로 삼고, 집계 결과의 합이 이 값과 같은지 검증하겠습니다.

### 한계와 추가 확인 사항
- 9991/9992의 정체, 미래 주문일 2건이 입력 오류인지 예약 주문인지는 원천 시스템 확인이 필요합니다.
- 2025-07과 2026-08은 첫 주문일(07-01)과 마지막 주문일(08-30)로 보아 거의 전체 월이지만, 2026-08-31 주문이 없는 것이 실제로 주문이 없었던 것인지 추출 시점 때문인지는 알 수 없습니다.

## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [37]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,96,4
2,orders,302,5,300,7
3,order_items,765,5,752,6


In [38]:
# [추가 검증] 전처리 전/후 비교: 행·열·결측·중복·변환 실패
conv_by_ds = conversion_summary.groupby('dataset')['conversion_failed'].sum()
before_after = pd.DataFrame([{
    'dataset': n,
    'rows_raw': len(raw_data[n]), 'rows_processed': len(processed_data[n]),
    'cols_raw': raw_data[n].shape[1], 'cols_processed': processed_data[n].shape[1],
    'missing_raw': int(raw_data[n].isna().sum().sum()), 'missing_processed': int(processed_data[n].isna().sum().sum()),
    'dup_raw': int(raw_data[n].duplicated().sum()), 'dup_processed': int(processed_data[n].duplicated().sum()),
    'conv_failed': int(conv_by_ds.get(n, 0)),
} for n in FILE_NAMES])
display(before_after)
for n, df in processed_data.items():
    na = df.isna().sum()
    print(f'processed 결측 {n}:', na[na > 0].to_dict() or '없음')
print('order_items 765 → 752: 중복 2 + 제외', len(removed_items), '=', 2 + len(removed_items))
print('products 102 → 96: 중복 2 + 제외', len(removed_products), '=', 2 + len(removed_products))
print('age 분포 변화: 41.0 고객 수', int((pd.to_numeric(customers.drop_duplicates()['age'], errors='coerce') == 41).sum()),
      '→', int((customers_clean['age'] == 41).sum()))

,dataset,rows_raw,rows_processed,cols_raw,cols_processed,missing_raw,missing_processed,dup_raw,dup_processed,conv_failed
0,customers,152,150,6,6,6,2,2,0,4
1,products,102,96,4,4,0,0,2,0,2
2,orders,302,300,5,7,0,9,2,0,3
3,order_items,765,752,5,6,0,0,2,0,7


processed 결측 customers: {'signup_date': 2}
processed 결측 products: 없음
processed 결측 orders: {'order_date': 3, 'order_month': 3, 'order_dayofweek': 3}
processed 결측 order_items: 없음
order_items 765 → 752: 중복 2 + 제외 11 = 13
products 102 → 96: 중복 2 + 제외 4 = 6
age 분포 변화: 41.0 고객 수 5 → 11


## 답안 5. 전처리 전/후 비교

| 항목 | 처리 전 (customers / products / orders / order_items) | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 152 / 102 / 302 / 765 | 150 / 96 / 300 / 752 | 중복 2행씩 제거. products는 price 변환 실패 2 + 0 이하 2 제외. order_items는 quantity(변환 실패 5, 0 이하 3)와 unit_price(변환 실패 2, 0 이하 1)로 11행 제외 |
| 열 수 | 6 / 4 / 5 / 5 | 6 / 4 / 7 / 6 | orders에 `order_month`, `order_dayofweek`, order_items에 `line_total` 추가 |
| 결측 | 6 / 0 / 0 / 0 | 2 / 0 / 9 / 0 | age·city는 대체로 0. 날짜 변환 실패를 NaT로 남겨 signup_date 2, order_date 3과 파생 컬럼 order_month 3·order_dayofweek 3이 생김 |
| 중복 | 2 / 2 / 2 / 2 | 0 / 0 / 0 / 0 | 키 중복이 모두 완전 중복이라 제거 |
| 변환 실패 | 보이지 않음 (문자열) | 16건 | customers 4 (age 2는 대체, signup_date 2는 NaT), orders 3 (NaT), products 2·order_items 7 (행 제외) |

![전처리 전후 비교](images/step05_before_after.png)

- Evidence: 17번 뒤 `[추가 검증]` 셀 Output

### 결과 관찰
- 행 수는 order_items 13행, products 6행, customers·orders 2행 줄었습니다.
- 결측은 customers에서 줄고 orders에서 늘었습니다.
- 값 분포도 바뀌었습니다. 41세 고객이 5명에서 11명이 되었습니다.

### 나의 해석과 판단
- 행 수 변화가 의도한 결과라고 판단한 근거는 줄어든 행을 원인별로 모두 설명할 수 있기 때문입니다. order_items는 2 + 5 + 3 + 2 + 1 = 13, products는 2 + 2 + 2 = 6으로 맞아떨어지고, 설명되지 않는 감소는 없습니다.
- orders 결측 증가(0 → 9)도 의도한 결과입니다. `2026-02-30` 같은 날짜를 문자열로 두면 오류가 숨어 있지만, NaT로 바꾸면 결측으로 드러나 셀 수 있습니다. order_month·order_dayofweek 6건은 같은 3개 주문에서 파생된 것이라 새로운 문제가 아닙니다.
- 다만 "기준대로 된 결과"와 "바람직한 결과"는 다릅니다. `3개`, `2개`, `5개` 3행은 복구할 수 있었는데 지워졌고, 상품 4개 제외는 연결 문제 24행을 만들었습니다. 이 두 가지는 기준의 부작용으로 판단합니다.

### 업무·분석적 의미
- 전후 비교표는 감사 추적(audit trail)입니다. "왜 주문 상세가 765행이 아니라 752행인가"라는 질문에 원인별 건수로 답할 수 있습니다.

### 한계와 추가 확인 사항
- 이 표는 행 수와 결측 수만 비교합니다. 대체로 생긴 분포 변화(41세 쏠림)나 관계 변화(FK 2 → 26)는 별도 셀에서 확인해야 보입니다.

## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [39]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('D:/dev/llm-data-analysis-course/data/processed/customers_clean.csv'),
 WindowsPath('D:/dev/llm-data-analysis-course/data/processed/orders_clean.csv'),
 WindowsPath('D:/dev/llm-data-analysis-course/data/processed/order_items_clean.csv'),
 WindowsPath('D:/dev/llm-data-analysis-course/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [40]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: D:\dev\llm-data-analysis-course\reports\ch05_preprocessing_summary.md


In [41]:
# [추가 검증] Notebook이 저장한 clean CSV 확인과 해시 기록
for path in sorted(PROCESSED_DIR.glob('*_clean.csv')):
    print(f'{path.name:24s} exists={path.exists()} size={path.stat().st_size:,} bytes')
notebook_hash = {f'{n}_clean.csv': md5(PROCESSED_DIR / f'{n}_clean.csv') for n in FILE_NAMES}

customers_clean.csv      exists=True size=5,924 bytes
order_items_clean.csv    exists=True size=24,671 bytes
orders_clean.csv         exists=True size=16,421 bytes
products_clean.csv       exists=True size=3,971 bytes


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [42]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [43]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(96, 4)
(300, 7)
(752, 6)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22096\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22096\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://p

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [44]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,order_items,765,5,752,6
2,orders,302,5,300,7
3,products,102,4,96,4


In [45]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [46]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.customer_id,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.product_id,26


In [47]:
# [추가 검증] 공식 단계별 셀 결과 == 20번 함수 결과 == src 모듈 결과
function_data = {'customers': customers_clean_func, 'products': products_clean_func,
                 'orders': orders_clean_func, 'order_items': order_items_clean_func}
for n in FILE_NAMES:
    pd.testing.assert_frame_equal(processed_data[n].reset_index(drop=True), module_processed_data[n].reset_index(drop=True))
    pd.testing.assert_frame_equal(function_data[n].reset_index(drop=True), module_processed_data[n].reset_index(drop=True))
print('단계별 셀 결과 == Notebook 함수 결과 == src.preprocessing 결과: PASS (4개 데이터 모두)')

단계별 셀 결과 == Notebook 함수 결과 == src.preprocessing 결과: PASS (4개 데이터 모두)


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


### STEP 14. 반복 실행 스크립트 실행 (가이드 Evidence)

가이드 명령을 그대로 실행한 결과와, 실패 원인을 고친 뒤 다시 실행한 결과를 모두 남겼습니다. 같은 명령을 PowerShell 터미널에서도 실행했습니다(아래 답안 6의 이미지). 마지막으로 Notebook 저장본, 스크립트 결과, 강사 저장소에 커밋된 결과를 비교했습니다.

In [48]:
# [추가 검증] STEP 14. 가이드 명령 그대로 실행 → 실패 원인 확인 → 수정 실행 → 결과 해시 비교
script = ['scripts/preprocess_data.py']
base_env = {k: v for k, v in os.environ.items() if k != 'PYTHONPATH'}
base_env['PYTHONIOENCODING'] = 'utf-8'

run1 = subprocess.run([sys.executable, *script], cwd=PROJECT_ROOT, env=base_env, capture_output=True, text=True, encoding='utf-8')
print('[1차] python scripts/preprocess_data.py → returncode', run1.returncode)
if run1.returncode:
    print('   ', run1.stderr.strip().splitlines()[-1])

run2 = subprocess.run([sys.executable, *script], cwd=PROJECT_ROOT, env={**base_env, 'PYTHONPATH': str(PROJECT_ROOT)},
                      capture_output=True, text=True, encoding='utf-8')
print('[2차] PYTHONPATH=<프로젝트 루트> python scripts/preprocess_data.py → returncode', run2.returncode)
print(run2.stdout)

script_hash = {name: md5(PROCESSED_DIR / name) for name in notebook_hash}
print('Notebook 저장본 vs 스크립트 결과:', {n: ('같음' if notebook_hash[n] == script_hash[n] else '다름') for n in notebook_hash})

# 공식 저장소에 커밋된 clean CSV와도 비교 (git 기준 변경 여부)
git_diff = subprocess.run(['git', 'status', '--short', '--', 'data/processed'], cwd=PROJECT_ROOT, capture_output=True, text=True).stdout.strip()
print('강사 저장소에 커밋된 data/processed 대비 변경:', git_diff or '없음 (공식 결과와 동일)')

[1차] python scripts/preprocess_data.py → returncode 1
    ModuleNotFoundError: No module named 'src'


[2차] PYTHONPATH=<프로젝트 루트> python scripts/preprocess_data.py → returncode 0
전처리 완료
입력 Raw 경로: practice\chapter05\data\raw

[전처리 전후 데이터 크기]
    dataset  rows_raw  columns_raw  rows_processed  columns_processed
  customers       152            6             150                  6
order_items       765            5             752                  6
     orders       302            5             300                  7
   products       102            4              96                  4

[중복 점검 결과]
    dataset  row_duplicate_count    key_column  key_duplicate_count
  customers                    0   customer_id                    0
   products                    0    product_id                    0
     orders                    0      order_id                    0
order_items                    0 order_item_id                    0

[파일 간 관계 점검 결과]
                                               check  invalid_count
  orders.customer_id exists in customers.customer_id              2
      o

강사 저장소에 커밋된 data/processed 대비 변경: 없음 (공식 결과와 동일)


### STEP 15. 저장 결과 다시 읽어 검증

In [49]:
# [추가 검증] STEP 15. 저장 결과 다시 읽어 검증
reread = {n: pd.read_csv(PROCESSED_DIR / f'{n}_clean.csv') for n in FILE_NAMES}
reread_rows = [{'dataset': n, 'shape': df.shape, 'columns_same': df.columns.tolist() == processed_data[n].columns.tolist(),
                'key_unique': df[key_columns[n]].is_unique} for n, df in reread.items()]
display(pd.DataFrame(reread_rows))

orders_check = reread['orders']
print('다시 읽은 order_date dtype:', orders_check['order_date'].dtype)
orders_check['order_date'] = pd.to_datetime(orders_check['order_date'], errors='coerce')
print('order_date 재변환 실패:', int(orders_check['order_date'].isna().sum()), '| order_month 결측:', int(orders_check['order_month'].isna().sum()))
print('quantity 저장 형식:', reread['order_items']['quantity'].dtype, reread['order_items']['quantity'].head(3).tolist())
print('line_total 합계 저장 전 == 재읽기:', processed_data['order_items']['line_total'].sum() == reread['order_items']['line_total'].sum())
print('customers_clean.csv에 name 컬럼 포함:', 'name' in reread['customers'].columns)

,dataset,shape,columns_same,key_unique
0,customers,"(150, 6)",True,True
1,products,"(96, 4)",True,True
2,orders,"(300, 7)",True,True
3,order_items,"(752, 6)",True,True


다시 읽은 order_date dtype: str
order_date 재변환 실패: 3 | order_month 결측: 3
quantity 저장 형식: float64 [2.0, 4.0, 3.0]
line_total 합계 저장 전 == 재읽기: True
customers_clean.csv에 name 컬럼 포함: True


### STEP 16. 전처리 보고서 검토

가이드 STEP 16의 확인 항목 9개에, 필요하다고 본 항목 2개(변환 실패로 제외된 행, invalid 관계 경고)를 더해 스크립트가 만든 보고서를 확인했습니다.

In [50]:
# [추가 검증] STEP 16. 스크립트가 만든 요약 보고서에 필요한 내용이 있는지 확인
report_text = (REPORT_DIR / 'ch05_preprocessing_summary.md').read_text(encoding='utf-8')
checks = {
    '전처리 결과 파일': 'customers_clean.csv', '전처리 전후 데이터 크기': '전처리 전후 데이터 크기',
    '중복 점검': '중복 점검', '관계 점검': '관계 점검', '문자열·숫자·날짜 처리 기준': '공백',
    '결측치 처리 기준': '중앙값', '이상값 처리 기준': '0 이하', '파생 컬럼': 'line_total', '업무 적용 시 한계': '주의 사항',
    '[추가] 변환 실패로 제외된 행': '변환 실패', '[추가] invalid 관계 경고·조치': 'invalid_count가 0',
}
for item, keyword in checks.items():
    print(f'{"있음" if keyword in report_text else "없음"}  {item}')

있음  전처리 결과 파일
있음  전처리 전후 데이터 크기
있음  중복 점검
있음  관계 점검
있음  문자열·숫자·날짜 처리 기준
있음  결측치 처리 기준
있음  이상값 처리 기준
있음  파생 컬럼
있음  업무 적용 시 한계
없음  [추가] 변환 실패로 제외된 행
없음  [추가] invalid 관계 경고·조치


## 답안 6. 재현 가능한 전처리 확인

- 생성된 clean CSV: `data/processed/customers_clean.csv` 5,924 bytes, `products_clean.csv` 3,971 bytes, `orders_clean.csv` 16,421 bytes, `order_items_clean.csv` 24,671 bytes
- 생성된 요약 보고서: `reports/ch05_preprocessing_summary.md`
- `python scripts/preprocess_data.py` 재실행 결과:
  - 가이드 명령을 그대로 실행하면 `ModuleNotFoundError: No module named 'src'`로 실패합니다(returncode 1). `python scripts/파일.py`로 실행하면 모듈 검색 경로의 첫 번째가 `scripts/` 폴더가 되어, 프로젝트 루트의 `src` 패키지를 찾지 못하기 때문입니다.
  - 프로젝트 루트를 `PYTHONPATH`에 추가하자 정상 종료했습니다(returncode 0). PowerShell에서는 프로젝트 루트에서 `$env:PYTHONPATH = "."` 후 `python scripts/preprocess_data.py`로 실행했습니다.
- 재실행 후 달라진 점: 없습니다.
  - Notebook 저장본과 스크립트 결과 4개 파일의 MD5 해시가 같습니다.
  - 공식 단계별 셀 결과, 20번 함수 결과, `src.preprocessing` 결과도 `assert_frame_equal`로 같습니다.
  - 강사 저장소에 커밋된 `data/processed` 파일과 비교해도 git 변경 사항이 없습니다. 공식 환경에서 만든 결과와 바이트 단위로 같다는 뜻입니다.
- 저장 결과 재확인(STEP 15): 4개 파일 모두 컬럼 동일, 키 고유. `order_date`는 문자열로 다시 읽혀 재변환이 필요했고, 재변환 실패는 저장 전과 같은 3건입니다. quantity는 `2.0`처럼 실수로 저장됐습니다. `customers_clean.csv`에는 `name` 컬럼이 남아 있습니다.
- 보고서 검토(STEP 16): 가이드 9개 항목은 모두 있습니다. 하지만 변환 실패로 제외된 행과 invalid 관계에 대한 경고는 없습니다.

![재실행 결과](images/step06_reproducible.png)

- Evidence: 22번 뒤 STEP 14·15·16 `[추가 검증]` 셀 Output, PowerShell 터미널 실행 결과 이미지

### 나의 해석과 판단
- Notebook에서 한 번 성공한 것은 "그때 그 순서로 셀을 실행했더니 됐다"는 뜻입니다. 필터 셀을 두 번 실행하거나 순서를 바꾸면 결과가 달라질 수 있고, 커널 상태는 남지 않습니다. 스크립트는 같은 입력이면 처음부터 같은 순서로 실행됩니다. 이번에는 세 가지 방법의 결과가 해시까지 같다는 것을 확인했기 때문에, 결과가 우연이 아니라는 근거가 있습니다.
- 재현 가능성에는 코드뿐 아니라 실행 방법과 실행 환경도 포함됩니다.
  - 실행 방법: 가이드 명령이 그대로는 실패했습니다. 스크립트 안에서 프로젝트 루트를 `sys.path`에 추가하는 방식(ch09·ch14 스크립트와 같은 방식)이 더 안전하다고 봅니다.
  - 실행 환경: 처음에 전역 Python(pandas 2.3.2)으로 같은 Notebook을 실행했을 때는, 날짜 변환에 실패한 3건의 `order_month`가 결측이 아니라 문자열 `"NaT"`로 저장됐습니다. 이 값은 `isna()` 점검을 통과해 월별 집계에서 가짜 월이 됩니다. 수업 지정 환경(`.venv`, pandas 3.0.5)에서는 정상적으로 결측이 되고, 그 결과가 강사 저장소에 커밋된 파일과 같습니다. 같은 코드라도 라이브러리 버전에 따라 결과 파일이 달라진다는 것을 직접 확인했습니다. Chapter 02에서 맞춘 `.venv`가 왜 필요한지 다시 느꼈습니다.
- 스크립트가 정상 종료했다고 데이터가 정상인 것은 아닙니다. invalid_count가 30행이어도 "전처리 완료"가 출력됩니다. 실행 성공 ≠ 분석 성공이므로 종료 코드가 아니라 출력된 검증 숫자를 읽어야 합니다.
- 보고서에 "0 이하 제외"만 있고 변환 실패 제외가 없으면, 읽는 사람은 9행이 다른 이유로 빠졌다는 것을 알 수 없습니다. 보고서는 실제로 한 일을 모두 적어야 한다고 생각합니다.
- `data/processed/*.csv`와 `reports/ch05_preprocessing_summary.md`는 강사 저장소의 실행 산출물이라 개인 저장소에는 올리지 않았습니다. 같은 환경에서 이 Notebook을 실행하면 같은 해시의 파일이 만들어집니다.

## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


### 23-1. 23번 예시 Prompt로 LLM 코드 검토

- 사용 LLM: Claude (Anthropic). 위 23번의 Prompt를 그대로 입력했습니다.
- LLM 답변 요약:
  1. 원본 직접 수정: `customers["age"] = ...`처럼 원본 변수에 바로 덮어씁니다. `customers_clean = customers.copy()`로 복사본을 만들어 처리해야 합니다.
  2. 결측 처리: 평균은 극단값에 민감합니다. 먼저 `pd.to_numeric(errors="coerce")`로 숫자로 바꾸고, 중앙값 대체 여부는 결측 원인을 확인한 뒤 정합니다. 대체했다면 대체 여부 컬럼을 남깁니다.
  3. 날짜 변환: 형식이 잘못된 값이 있으면 오류가 납니다. `errors="coerce"`를 쓰고 실패 건수(`isna().sum()`)를 확인합니다.
  4. 중복 제거: `drop_duplicates()`는 전체 행 중복만 지웁니다. 키(`customer_id`) 중복을 따로 확인하고, 다른 값을 가진 키 중복은 자동 삭제하지 않습니다.
  5. `line_total`: quantity·unit_price가 문자열이면 계산이 안 되거나 잘못됩니다. 쉼표 제거 후 숫자로 바꾸고 변환 실패를 확인합니다.
  6. 수정 코드: 복사본 → 숫자·날짜 coerce 변환 → 실패 건수 출력 → 중앙값 대체 + 플래그 → 키 중복 확인 후 제거 → 숫자 변환 후 `line_total`
- 아래 셀에서 예시 코드를 Chapter 05 Raw 복사본에 그대로 실행해 LLM의 지적이 실제로 맞는지 확인했습니다.

In [51]:
# [추가 검증] 23번 예시 코드를 Chapter 05 Raw 복사본에 그대로 실행하면?
trial = {n: df.copy() for n, df in raw_data.items()}
for label, code in [
    ('age.fillna(mean)', lambda: trial['customers']['age'].fillna(trial['customers']['age'].mean())),
    ('pd.to_datetime(order_date)', lambda: pd.to_datetime(trial['orders']['order_date'])),
    ('quantity * unit_price', lambda: trial['order_items']['quantity'] * trial['order_items']['unit_price']),
]:
    try:
        result = code()
        print(f'{label}: 실행됨 → 결과 dtype {result.dtype}, 예시 {result.head(2).tolist()}')
    except Exception as e:
        print(f'{label}: {type(e).__name__} - {str(e).splitlines()[0][:90]}')
print('drop_duplicates() 후 customer_id 중복:', int(trial['customers'].drop_duplicates()['customer_id'].duplicated().sum()),
      '(이번 데이터는 키 중복이 모두 완전 중복이라 결과상 문제 없음)')

age.fillna(mean): TypeError - Cannot perform reduction 'mean' with string dtype
pd.to_datetime(order_date): ValueError - time data "not-a-date" doesn't match format "%Y-%m-%d". You might want to try:
quantity * unit_price: TypeError - Can only string multiply by an integer.
drop_duplicates() 후 customer_id 중복: 0 (이번 데이터는 키 중복이 모두 완전 중복이라 결과상 문제 없음)


- 확인 결과: LLM이 지적한 3가지가 모두 실제로 발생했습니다. 평균 대체는 `TypeError`(문자열 컬럼에 mean 불가), 날짜 변환은 `ValueError`(`not-a-date`), `line_total`은 `TypeError`(문자열 곱셈)가 났습니다. 중복 제거는 이번 데이터에서는 키 중복이 모두 완전 중복이라 결과상 문제가 없었습니다. 그래서 4번 지적은 "이번 데이터에서는 해당 없음, 일반 원칙으로는 채택"으로 판단했습니다.
- 수정 코드의 순서(복사본 → 변환 → 실패 건수 → 대체 → 중복 → 파생)는 공식 Notebook 6~15번 흐름과 같아 채택했습니다. 단, LLM 답변도 FK 관계 재검증은 언급하지 않았습니다. 이 부분은 사람이 추가해야 하는 단계입니다.

### 23-2. 가이드 STEP 17 Prompt로 결측 처리 기준 검토

원본 고객 행과 이름은 제공하지 않고 구조와 품질 요약 숫자만 제공했습니다.

```text
customers 데이터 품질 요약 (실제 고객 행은 제공하지 않음)
- rows: 152 (전체 행 중복 2행 포함, 중복 제거 시 150)
- customer_id missing/duplicate: 0 / 2 (2건 모두 전체 행 중복)
- age: 문자열 타입. 결측 4, 'unknown' 문자열 2, 최솟값 18, 최댓값 150, 중앙값 41, 평균 약 43.9
- city: 결측 2, 앞뒤 공백 표기 5건
- gender: F/M, 결측 0

age 결측치 처리 후보를 비교해 주세요.
각 방법의 장단점과 처리 전후 검증 항목을 제안해 주세요.
실제 업무 의미를 확인하지 않은 값은 단정하지 마세요.
```

LLM 답변 요약:

| 후보 | 장점 | 단점 |
| --- | --- | --- |
| 행 삭제 | 단순하고 가짜 값을 만들지 않음 | 고객 6명(약 4%)이 빠지고, 조인 방식에 따라 그 고객의 주문까지 분석에서 빠질 수 있음 |
| 전체 중앙값 대체 | 150 같은 극단값에 강하고 단순함 | 분산이 줄고 특정 나이(41)에 값이 몰려 연령대 분석이 왜곡됨 |
| 그룹별 중앙값 (예: 성별) | 그룹 차이를 반영 | 그룹 표본이 작으면 불안정하고, 그룹과 나이의 관계에 대한 근거가 필요함 |
| 결측 유지 + 플래그 / "연령 미상" 구간 | 정보 손실이 없고 나중에 방법을 바꿀 수 있음 | 수치 계산마다 결측 처리를 다시 해야 함 |
| 모델 기반 대체 (KNN, 회귀) | 다른 변수 정보를 활용 | 150행 규모와 설명 변수 부족을 고려하면 과함 |

- 권장 순서: ① `'unknown'`과 NaN을 같은 결측으로 통일하고 건수를 기록합니다. ② 150은 대체 전에 이상값 후보로 분리하고, 오류인지는 업무 확인 후 결정합니다. ③ 연령대 분석이 목적이면 결측 유지 + 플래그, 단순 요약이 목적이면 중앙값 + 플래그(`age_imputed`)를 씁니다.
- 처리 전후 검증 항목: 결측 수(6 → 0), 대체 건수, 평균·중앙값·분포 비교, 연령대별 고객 수 비교, 중복 제거 후 계산했는지, 원본 미변경 여부
- 단정하지 않을 것: 150이 입력 오류인지, `'unknown'`이 응답 거부인지 시스템 기본값인지

아래 셀에서 LLM 제안 중 데이터로 확인할 수 있는 것을 모두 계산했습니다.

In [52]:
# [추가 검증] STEP 17 LLM 제안을 실제 데이터로 확인
age_dedup = pd.to_numeric(customers.drop_duplicates()['age'], errors='coerce')
print('중앙값  실습 기준(중복·150 포함):', age_median, '| 중복 제거:', age_dedup.median(), '| 중복 제거 + 150 결측:', age_dedup.mask(age_dedup > 120).median())
print('평균    중복 제거:', round(age_dedup.mean(), 2), '| 중복 제거 + 150 결측:', round(age_dedup.mask(age_dedup > 120).mean(), 2))
print('성별 중앙값:', customers.drop_duplicates().assign(age=age_dedup).groupby('gender')['age'].median().to_dict())
bins = [0, 29, 39, 49, 59, 200]
labels = ['20대 이하', '30대', '40대', '50대', '60대 이상']
dist = pd.DataFrame({
    '결측 유지': pd.cut(age_dedup, bins, labels=labels).value_counts(dropna=False).sort_index(),
    '중앙값 대체': pd.cut(age_dedup.fillna(age_dedup.median()), bins, labels=labels).value_counts(dropna=False).sort_index(),
})
display(dist.T)
drop_ids = customers.drop_duplicates().loc[age_dedup.isna(), 'customer_id']
print('행 삭제안 적용 시 고객이 사라지는 주문(FK 깨짐):', int(orders_clean['customer_id'].isin(drop_ids).sum()), '건')

중앙값  실습 기준(중복·150 포함): 41.0 | 중복 제거: 41.0 | 중복 제거 + 150 결측: 41.0
평균    중복 제거: 44.03 | 중복 제거 + 150 결측: 43.29
성별 중앙값: {'F': 40.0, 'M': 44.0}


age,20대 이하,30대,40대,50대,60대 이상,NaN
결측 유지,35.0,31.0,24.0,23.0,31.0,6.0
중앙값 대체,35.0,31.0,30.0,23.0,31.0,NaN


행 삭제안 적용 시 고객이 사라지는 주문(FK 깨짐): 9 건


### 23-3. 사람이 확인한 결과

| 확인 기준 (가이드 STEP 17) | LLM 답변 | 데이터 확인 결과 | 판단 |
| --- | --- | --- | --- |
| raw를 덮어쓰는가? | 원본 미변경을 검증 항목에 포함 | raw MD5 실행 전후 동일 (마지막 Evidence 셀) | 채택 |
| 변환 실패를 확인하는가? | `'unknown'`을 결측으로 통일하고 건수 기록 | 원래 결측 4, 변환 실패 2를 분리해 기록 | 채택 |
| 결측 처리 근거가 있는가? | 목적별 선택 | 중앙값은 중복·150 처리와 관계없이 41.0. 평균은 44.03 → 43.29로 변함 | 채택. 중앙값이 기준 선택에 안정적이라는 근거 확보 |
| 키 중복을 자동 삭제하지 않는가? | "중복 제거 후 계산"만 언급 | 키 중복 8건 모두 완전 중복 (`key_dup_not_full_dup` = 0) | 보완. 키 충돌 확인 후에만 제거 |
| 이상값을 자동 삭제하지 않는가? | 150 분리, 업무 확인 후 결정 | 150은 고객 33 한 명, 주문 2건. 중앙값에는 영향 없음 | 채택 |
| 관계를 재검증하는가? | 언급 없음 | 행 삭제안을 쓰면 고객이 사라지는 주문 9건 발생 | 누락. 사람이 추가 |
| 변경 내역을 기록하는가? | `age_imputed` 플래그 | 중앙값 대체 시 40대가 24명 → 30명 | 채택하되 공식 함수에 플래그가 없어 저장 결과에는 미적용 (한계) |

### 나의 해석과 판단
- 이번 데이터에서는 중앙값이 어떤 조건에서도 41.0이었습니다. "150이 결측 대체값을 왜곡한다"는 걱정은 중앙값에는 해당하지 않고 평균에만 해당한다는 것을 숫자로 확인했습니다. LLM 답변은 방향은 맞지만, 이 데이터에서 실제로 차이가 나는지는 계산해야 알 수 있었습니다.
- 성별 중앙값(F 40, M 44)은 차이가 있지만, 성별과 나이의 관계를 설명할 업무 근거가 없어 그룹별 대체는 채택하지 않았습니다. 실습 기준(전체 중앙값 41)을 유지하고, 대체 표시가 없다는 점을 한계로 남겼습니다.
- 두 번의 LLM 답변 모두 테이블 하나 안의 처리만 이야기했고 다른 테이블과의 관계는 언급하지 않았습니다. 이번 실습에서 가장 큰 문제가 FK였다는 점을 생각하면, LLM 답변은 질문 범위 밖의 영향을 사람이 따로 확인해야 합니다.

## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [53]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.
missing_table = pd.concat(
    [missing_summary(df).reset_index(names='column').assign(dataset=name) for name, df in processed_data.items()],
    ignore_index=True,
)[['dataset', 'column', 'missing_count', 'missing_ratio']]
display(missing_table[missing_table['missing_count'] > 0])
print('결측이 남은 컬럼 수:', int((missing_table['missing_count'] > 0).sum()), '/ 전체', len(missing_table))

,dataset,column,missing_count,missing_ratio
0,customers,signup_date,2,1.33
10,orders,order_date,3,1.00
11,orders,order_month,3,1.00
12,orders,order_dayofweek,3,1.00


결측이 남은 컬럼 수: 4 / 전체 23


In [54]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
age_out = customers_clean.loc[(customers_clean['age'] < 18) | (customers_clean['age'] > 100), ['customer_id', 'age']]
print('18세 미만:', int((customers_clean['age'] < 18).sum()), '/ 100세 초과:', int((customers_clean['age'] > 100).sum()))
display(age_out)

18세 미만: 0 / 100세 초과: 1


,customer_id,age
32,33,150.0


In [55]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.
monthly_orders = orders_clean.groupby('order_month', dropna=False).size().rename('order_count')
print(monthly_orders.to_string())
print('합계:', int(monthly_orders.sum()), '/ orders_clean', len(orders_clean))

order_month
2025-07    19
2025-08    26
2025-09    12
2025-10    22
2025-11    23
2025-12    23
2026-01    26
2026-02    19
2026-03    22
2026-04    26
2026-05    20
2026-06    24
2026-07    12
2026-08    21
2027-01     1
2028-12     1
NaN         3
합계: 300 / orders_clean 300


In [56]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.
top10 = order_items_clean.nlargest(10, 'line_total')[['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']]
top10['order_status'] = top10['order_id'].map(status_lookup)
display(top10)

,order_item_id,order_id,product_id,quantity,unit_price,line_total,order_status
750,751,295,31,100.0,155000.0,15500000.0,refunded
737,738,290,96,50.0,96000.0,4800000.0,completed
319,320,129,23,5.0,197000.0,985000.0,completed
9,10,5,65,5.0,193000.0,965000.0,cancelled
20,21,9,36,5.0,193000.0,965000.0,completed
24,25,11,65,5.0,193000.0,965000.0,completed
116,117,51,65,5.0,193000.0,965000.0,completed
293,294,119,36,5.0,193000.0,965000.0,refunded
290,291,117,47,5.0,188000.0,940000.0,cancelled
10,11,6,97,5.0,187000.0,935000.0,completed


### 24번 실습 과제 답안

- 과제 1: 결측이 남은 컬럼은 23개 중 4개입니다. customers.signup_date 2 (1.33%), orders.order_date·order_month·order_dayofweek 각 3 (1.00%)이고, 모두 날짜 변환 실패에서 왔습니다.
- 과제 2: 18세 미만 0명, 100세 초과 1명(customer_id 33, 150세)입니다. 실습 기준은 0 이하만 제외하므로 남아 있습니다.
- 과제 3: 2025-07 ~ 2026-08은 월 12~26건입니다. 그 외 `2027-01` 1건, `2028-12` 1건, 결측 3건이 있고, 합계 300건이 orders_clean 행 수와 같습니다. 이 5건은 Chapter 06 분석 기간(2025-07-01 ~ 2026-08-30)에서 제외합니다.
- 과제 4: 1위(15,500,000, quantity 100)는 refunded, 2위(4,800,000, quantity 50)는 completed 주문입니다. 3위(985,000)부터는 quantity 5 × 고가 상품입니다. 상위 10개 중 completed는 6개입니다.
- 과제 5: 4개 데이터셋 모두 행 수가 줄었습니다. 원인별 내역은 답안 5에 정리했습니다(customers·orders 중복 2, products 중복 2 + 제외 4, order_items 중복 2 + 제외 11).
- 과제 6: 결측치 처리 기준 검토 Prompt는 23-2에 있습니다. 실제 고객 행 대신 요약 숫자만 넣고, "업무 의미를 확인하지 않은 값은 단정하지 말 것"을 조건으로 넣었습니다.

## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.


## 내가 제안하는 전처리 기준 (비교만, 저장하지 않음)

저장 결과는 Chapter 06이 공통으로 쓰는 `data/processed`와 맞추기 위해 공식 기준 그대로 두었습니다. 대신 아래 셀에서 내가 판단한 기준을 적용하면 무엇이 달라지는지 비교했습니다.

| 제안 기준 | 근거 (이 Notebook에서 확인한 것) |
| --- | --- |
| quantity의 `개`를 제거한 뒤 숫자 변환 | `3개`, `2개`, `5개`는 의미가 분명함. 그중 `2개`는 completed 주문 42의 행 |
| 변환 실패·0 이하 행은 지우지 않고 `hold` 표시 | 원인별 건수가 남고, 주문 175처럼 상세 행이 사라지는 주문이 생기지 않음 |
| products는 행을 지우지 않고 `price_valid` 표시 | 정상 상품 726행은 단가 = 마스터 가격, 이상 상품 5개는 단가가 일정해 마스터 오류로 판단. `line_total`은 unit_price로 계산하므로 상품을 지울 이유가 없음 |
| category 글자 사이 공백 제거 | `뷰 티` → `뷰티`로 집계 단위 통일 |

In [57]:
# [추가 검증] 내가 제안하는 기준을 적용하면 무엇이 달라지는가 (저장하지 않고 비교만)
alt_items = strip_string_columns(order_items).drop_duplicates()
alt_items['quantity'] = to_number(alt_items['quantity'].astype(str).str.replace('개', '', regex=False))
alt_items['unit_price'] = to_number(alt_items['unit_price'])
alt_items['hold'] = (alt_items['quantity'].isna() | (alt_items['quantity'] <= 0)
                     | alt_items['unit_price'].isna() | (alt_items['unit_price'] <= 0))
alt_products = strip_string_columns(products).drop_duplicates()
alt_products['category'] = alt_products['category'].str.replace(' ', '', regex=False)
alt_products['price'] = to_number(alt_products['price'])
alt_products['price_valid'] = alt_products['price'].between(1, 1_000_000)
valid_items = alt_items[~alt_items['hold']]

alt_compare = pd.DataFrame([
    {'항목': '분석 가능한 order_items 행', '실습 기준': len(order_items_clean), '제안 기준': len(valid_items)},
    {'항목': '보류(hold) 표시 order_items 행', '실습 기준': 0, '제안 기준': int(alt_items['hold'].sum())},
    {'항목': 'products 행 (price_valid=False 포함)', '실습 기준': len(products_clean), '제안 기준': len(alt_products)},
    {'항목': 'category 고유값', '실습 기준': products_clean['category'].nunique(), '제안 기준': alt_products['category'].nunique()},
    {'항목': '없는 product_id 참조 행', '실습 기준': len(invalid_products),
     '제안 기준': int((~valid_items['product_id'].isin(alt_products['product_id'])).sum())},
    {'항목': '상세 행이 사라진 주문', '실습 기준': len(orders_without_items),
     '제안 기준': int((~orders_clean['order_id'].isin(alt_items['order_id'])).sum())},
])
display(alt_compare)
print('복구된 quantity 원본 값:', order_items.loc[order_items['quantity'].astype(str).str.endswith('개'), 'quantity'].tolist())
print('price_valid=False 상품:', alt_products.loc[~alt_products['price_valid'], 'product_id'].tolist())

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22096\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22096\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://p

,항목,실습 기준,제안 기준
0,분석 가능한 order_items 행,752,755
1,보류(hold) 표시 order_items 행,0,8
2,products 행 (price_valid=False 포함),96,100
3,category 고유값,10,7
4,없는 product_id 참조 행,26,2
5,상세 행이 사라진 주문,1,0


복구된 quantity 원본 값: ['3개', '2개', '5개']
price_valid=False 상품: [18, 48, 73, 95, 99]


### 제안 기준 비교 결과
- 분석 가능한 order_items가 752행 → 755행으로 늘었습니다(`3개`, `2개`, `5개` 복구). 나머지 8행은 지우지 않고 `hold`로 남겨 원인별로 다시 판단할 수 있습니다.
- products 100행이 모두 남고(5개는 `price_valid=False`), 없는 product_id 참조가 26행 → 2행(원본부터 없던 9991, 9992)으로 줄었습니다. 상세 행이 사라진 주문도 1건 → 0건이 됩니다.
- category는 7종으로 정리됩니다. 상품명과 category가 다른 5개 상품은 공백 제거로 해결되지 않아 담당자 확인이 필요합니다.

## 답안 7. Chapter 05 최종 판단

### 내가 정한 전처리 원칙 3가지
1. 지우기 전에 원인별로 셉니다. 변환 실패, 0 이하, 중복처럼 성격이 다른 문제를 필터 하나로 지우지 않고, 원인별 건수를 보고서에 남깁니다.
2. 다른 테이블이 참조하는 기준(마스터) 테이블의 행은 지우지 않고 표시합니다. 이번에 상품 4개를 지운 것이 정상 주문 상세 24행의 연결을 끊었습니다.
3. 전처리 후에는 관계·파생 컬럼·저장 파일을 다시 검증하고, "완료" 메시지가 아니라 검증 숫자로 판단합니다. 실행 환경(pandas 버전)까지 같아야 같은 결과라고 말할 수 있습니다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
`df[df['quantity'] > 0]` 같은 비교 필터입니다.
- `NaN > 0`이 False라서 변환 실패 행까지 조용히 지웁니다. 이번에 9행이 0 이하가 아닌데도 0 이하 기준으로 빠졌고, 그중 4행은 completed 주문의 행이었습니다.
- 상품 마스터에 적용하면 그 상품을 산 정상 주문 상세 24행이 연결을 잃고, 카테고리별 completed 상품 주문 금액에서 4,015,000이 빠집니다.
- 그래도 오류가 나지 않고, 스크립트는 "전처리 완료"를 출력하며, 보고서에는 "0 이하 제외"로만 남습니다. 문제를 만들면서 그 흔적까지 지우는 처리라서 가장 위험하다고 판단했습니다.

### 다음 EDA에서 특히 주의할 데이터 특성
- 분석 기간: 2025-07-01 ~ 2026-08-30로 고정합니다. 주문일 결측 3건(그중 completed 2건)과 미래 주문 2건(2027-01-15, 2028-12-31)은 제외하고 제외 건수를 기록합니다.
- 기준 합계: completed + 기간 + category 연결 기준 주문 176건, 421행, 상품 주문 금액 134,513,000. 월별·카테고리별 집계의 합계가 이 값과 같은지 검증합니다.
- 카테고리: `전자 기기`, `생활 용품`, `뷰 티` 표기 변형과 상품명-category 불일치 5개 상품이 있습니다. 집계 전에 공백을 정리하고 불일치 상품 목록을 함께 보고합니다.
- merge: 없는 product_id 26행, 없는 customer_id·order_id 각 2건이 있습니다. merge 전후 행 수, `_merge` 결과, 합계를 확인합니다.
- 극단값: completed의 quantity 50 한 행(주문 290, 4,800,000)이 기준 합계 134,513,000의 3.57%입니다. 이 행을 포함/제외한 결과를 함께 봅니다.
- 고객 분석을 할 경우: 주문일이 가입일보다 앞선 주문 58건(20.1%)이 있어 가입일 기준 분석은 보류합니다. age는 41세에 대체값 6명이 몰려 있고 150세 1명이 남아 있습니다.
- 용어: `line_total` 합계는 상품 주문 금액입니다. completed만 걸러도 할인·배송비·세금·실제 결제액 정보가 없으므로 회계상 매출이라고 부르지 않습니다.

### 현재 전처리의 한계
- 반품 표기 규칙, 대량 주문 허용 범위, 0원 상품 정책, 9991/9992의 정체, `signup_date`의 정의는 데이터만으로 확인할 수 없어 담당자 확인이 필요합니다.
- 대체 표시(`age_imputed`)와 보류 표시(`hold`, `price_valid`)가 저장 결과에는 없습니다. 제안 기준은 비교만 하고 저장하지 않았습니다.
- 실습 기준의 이상값 정의가 "0 이하"뿐이라 age 150, price 9,999,999, quantity 50·100은 저장 결과에 그대로 있습니다.
- 합성 데이터라 실제 운영 데이터의 문제(인코딩, 시간대, 부분 환불 등)는 다루지 못했습니다.

In [58]:
# [추가 검증] Chapter 05 완료 Evidence 자동 점검
raw_hash_end = {name: md5(RAW_DIR / f'{name}.csv') for name in FILE_NAMES}
evidence = {
    'Notebook 실행 (.venv 커널, pandas 3.0.5)': sys.executable.endswith(os.path.join('.venv', 'Scripts', 'python.exe')) and pd.__version__ == '3.0.5',
    'raw 원본 보존 (실행 전후 MD5 동일)': raw_hash_start == raw_hash_end,
    'processed CSV 4개 생성 (크기 > 0)': all((PROCESSED_DIR / f'{n}_clean.csv').stat().st_size > 0 for n in FILE_NAMES),
    '숫자/날짜 변환 실패 점검': int(conversion_summary['conversion_failed'].sum()) == 16,
    '전체 중복/키 중복 점검 (키 충돌 0)': bool((duplicate_check['key_dup_not_full_dup'] == 0).all()),
    'PK/FK 관계 검증 실행 + 원인 분리': len(relationship_checks) == 3 and len(product_cause) > 0,
    '파생 컬럼 확인 (line_total 재계산 일치)': bool((recalc == order_items_clean['line_total']).all()),
    'preprocess_data.py 재실행 (returncode 0)': run2.returncode == 0,
    'Notebook 결과 == 스크립트 결과 (MD5)': notebook_hash == script_hash,
    'ch05_preprocessing_summary.md 생성': (REPORT_DIR / 'ch05_preprocessing_summary.md').exists(),
    '저장 결과 재확인 (컬럼·키)': all(r['columns_same'] and r['key_unique'] for r in reread_rows),
}
for item, ok in evidence.items():
    print(f'{"PASS" if ok else "FAIL"}  {item}')
print('\n참고: PK/FK invalid_count =', relationship_checks['invalid_count'].tolist(), '→ 검증은 PASS, 관계 자체는 불완전 (4·7번 참고)')

PASS  Notebook 실행 (.venv 커널, pandas 3.0.5)
PASS  raw 원본 보존 (실행 전후 MD5 동일)
PASS  processed CSV 4개 생성 (크기 > 0)
PASS  숫자/날짜 변환 실패 점검
PASS  전체 중복/키 중복 점검 (키 충돌 0)
PASS  PK/FK 관계 검증 실행 + 원인 분리
PASS  파생 컬럼 확인 (line_total 재계산 일치)
PASS  preprocess_data.py 재실행 (returncode 0)
PASS  Notebook 결과 == 스크립트 결과 (MD5)
PASS  ch05_preprocessing_summary.md 생성
PASS  저장 결과 재확인 (컬럼·키)

참고: PK/FK invalid_count = [2, 2, 26] → 검증은 PASS, 관계 자체는 불완전 (4·7번 참고)


## Chapter 05 완료 Evidence

- Notebook 실행: PASS (`.venv` 커널, pandas 3.0.5, 강사 저장소 공식 clone)
- raw 원본 보존 확인: PASS (실행 전후 4개 파일 MD5 동일)
- processed CSV 4개 생성: PASS
- 숫자/날짜 변환 실패 점검: PASS (16건, 실패한 원본 값 목록 포함)
- 전체 중복/키 중복 점검: PASS (키 중복 8건 모두 완전 중복)
- PK/FK 관계 검증: PASS (검증 실행, 원인 분리). 단, invalid 2 / 2 / 26행이 남아 관계가 완전하지는 않습니다.
- 파생 컬럼 확인: PASS (`line_total` 재계산 일치, `order_month` 결측 3건 = 날짜 변환 실패 3건)
- preprocess_data.py 재실행: PASS (가이드 명령 그대로는 `ModuleNotFoundError`, `PYTHONPATH` 지정 후 성공)
- ch05_preprocessing_summary.md 생성: PASS
- 저장 결과 재확인: PASS (Notebook = 스크립트 = 강사 저장소 커밋 결과)

### 내가 적용한 전처리 기준
- 저장 결과: 공식 실습 기준(문자열 strip, 숫자·날짜 coerce 변환, age 중앙값 41.0, city `Unknown`, 상태값 3종 표준화, 완전 중복 제거, price·quantity·unit_price 0 이하 제외, 파생 컬럼 3개)
- 추가 검증: 변환 실패와 0 이하를 원인별로 분리, 이상값 후보를 주문 상태·주문 단가와 대조, FK 문제를 "원본부터 없음"과 "전처리로 생김"으로 분리, Chapter 06 분석 범위를 날짜·건수·합계로 확정

### 전처리 전후 달라진 점
- 행 수 152 / 102 / 302 / 765 → 150 / 96 / 300 / 752, 열 수 orders +2, order_items +1
- 중복 8행 → 0, 결측 6 → 11 (age·city 대체로 감소, 날짜 NaT와 파생 컬럼으로 증가)
- product_id FK 문제 2행 → 26행, 상세 행 없는 주문 0 → 1건 (전처리로 생긴 부작용)

### 아직 업무 확인이 필요한 항목
- 9991/9992 고객·주문·상품의 정체, 미래 주문일 2건(256, 287)
- quantity -1 (refunded 주문 241)이 반품 표기 규칙인지, quantity 50·100이 허용되는 대량 주문인지
- 상품 18, 48, 73, 95, 99의 올바른 마스터 가격 (주문 단가는 각각 140,000 / 25,000 / 156,000 / 82,000 / 187,000으로 일정)
- 상품명과 category가 다른 5개 상품(3, 27, 44, 70, 78)의 올바른 category
- `signup_date`의 정의 (주문일보다 늦은 가입일 58건)

## 최종 제출 체크

- [x] raw 원본을 덮어쓰지 않았습니다. (실행 전후 MD5 동일)
- [x] 처리 전/후 비교를 남겼습니다.
- [x] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [x] clean CSV와 재실행 결과를 확인했습니다.
- [x] 개인정보/Secret이 없습니다. (`customers.name` 미출력, API Key 없음)
- [x] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.